# 🔬 TopoActin V3 - Complete Interactive Preview

## **Unified: Full 3D Processing + Interactive Visualization**

<div style="background-color: #e8f4f8; padding: 20px; border-radius: 10px; border-left: 5px solid #2196F3;">

### ✨ **Complete Feature Set**

#### 🔬 **V3 Advanced Processing**
- **3D Z-Stack Support** - Full 4D [Z,C,Y,X] with 3D Sato filter
- **Rescue Mode** - Prevents filtering paradox (keeps "sick" cells)
- **3-Zone Topology** - Edge/middle/center fortification analysis
- **CLAHE Enhancement** - Adaptive histogram equalization
- **Cellpose v4** - Latest cpsam model

#### 🎨 **Interactive Visualization**
- **🔍 Preview Buttons** - Click to see changes at EVERY step
- **Real-Time Feedback** - Adjust → Preview → Iterate
- **6-Panel Viz** - Complete pipeline visualization
- **Quality Stats** - Live filtering with rescue tracking

</div>

---

### 📋 **10-Step Interactive Workflow**

| Step | Stage | Features |
|------|-------|----------|
| **1** | 📁 Load Image | Auto-detect 2D/3D/4D formats |
| **2** | 🎨 Brightness | Manual + CLAHE modes |
| **3** | 🔬 Segmentation | Cellpose v4 (cpsam) |
| **4** | 🧬 Nucleus Filter | **Rescue Mode** (V3) |
| **5** | 🎯 Cell Filter | Border/size/quality |
| **6** | 🦴 Skeletonization | 2D/3D Sato filters |
| **7** | 🧬 Topology | 3-zone analysis |
| **8** | 📊 Summary | Filtering stats |
| **9** | ⚙️ Batch Config | Multi-experiment |
| **10** | 💾 Export | Save config_v3.json |

---

**📂 Directory**: `C:\work\TopoActin_V3_3D`  
**🎯 Each step**: Sliders → 🔍 Preview button → See results!

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings('ignore')

# Core packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
import tifffile
from pathlib import Path
import pandas as pd
import json
import gc
from collections import Counter

# Jupyter widgets
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Image processing
from scipy import ndimage
from skimage.morphology import (skeletonize, remove_small_objects, disk, 
                                binary_closing, binary_erosion)
from skimage import measure, exposure  # exposure for CLAHE
from skan import Skeleton, summarize

# Cellpose
from cellpose import models, core

# V3 pipeline modules
import sys
sys.path.append('.')
try:
    from filtering import NucleiFilter, CellFilter
    print('✅ Imported V3 filtering modules')
except ImportError:
    print('⚠️  filtering.py not found - place it in same directory')

# GPU check
gpu_available = core.use_gpu()
print(f'GPU: {"✅ Available" if gpu_available else "❌ Not available (CPU mode)"}')

print('\n' + '='*80)
print(' '*15 + '🔬 TopoActin V3 - COMPLETE INTERACTIVE PREVIEW')
print('='*80)
print('✨ All V3 features | Interactive buttons | Real-time preview')
print('='*80 + '\n')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS (V3)
# ═══════════════════════════════════════════════════════════════════════════════

def sato_tubeness_3d(volume, sigmas):
    """3D Sato filter for Z-stack fiber detection"""
    responses = []
    for sigma in sigmas:
        Hxx = ndimage.gaussian_filter(volume, sigma, order=[2, 0, 0])
        Hyy = ndimage.gaussian_filter(volume, sigma, order=[0, 2, 0])
        Hzz = ndimage.gaussian_filter(volume, sigma, order=[0, 0, 2])
        tubeness = -np.minimum(Hxx, np.minimum(Hyy, Hzz))
        tubeness = np.maximum(tubeness, 0)
        responses.append(tubeness * sigma**2)
    return np.max(responses, axis=0)

def sato_tubeness_2d(image, sigmas):
    """2D Sato filter for fiber detection"""
    responses = []
    for sigma in sigmas:
        Hxx = ndimage.gaussian_filter(image, sigma, order=[2, 0])
        Hxy = ndimage.gaussian_filter(image, sigma, order=[1, 1])
        Hyy = ndimage.gaussian_filter(image, sigma, order=[0, 2])
        tmp = np.sqrt((Hxx - Hyy)**2 + 4*Hxy**2)
        lambda2 = 0.5 * (Hxx + Hyy - tmp)
        tubeness = np.zeros_like(image, dtype=np.float32)
        mask = lambda2 < 0
        tubeness[mask] = np.abs(lambda2[mask])
        responses.append(tubeness * sigma**2)
    return np.max(responses, axis=0)

def adjust_brightness_contrast(image, brightness=1.0, contrast=1.0, gamma=1.0):
    """Adjust image brightness/contrast/gamma"""
    img = image.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-10)
    img = img * brightness
    mean = img.mean()
    img = (img - mean) * contrast + mean
    img = np.power(np.clip(img, 0, 1), 1.0 / gamma)
    return (img * 65535).astype(np.uint16)

def apply_clahe(image, clip_limit=0.03, tile_size=8):
    """Apply image-level CLAHE. tile_size controls the local region (kernel_size in skimage)."""
    img_norm = (image.astype(np.float32) - image.min()) / (image.max() - image.min() + 1e-10)
    enhanced = exposure.equalize_adapthist(img_norm, kernel_size=(tile_size, tile_size),
                                           clip_limit=clip_limit, nbins=256)
    return (enhanced * 65535).astype(np.uint16)

def extract_cell_region(cell_mask, image, padding=10):
    """Extract and crop cell region"""
    rows, cols = np.where(cell_mask)
    if len(rows) == 0:
        return None, None, None
    r0, r1 = max(0, rows.min()-padding), min(image.shape[0], rows.max()+1+padding)
    c0, c1 = max(0, cols.min()-padding), min(image.shape[1], cols.max()+1+padding)
    cell_image = image[r0:r1, c0:c1].copy() * cell_mask[r0:r1, c0:c1]
    return cell_image, cell_mask[r0:r1, c0:c1].copy(), (r0, c0, r1, c1)

print('✅ V3 Helper functions loaded')
print('  - sato_tubeness_3d() - 3D fiber detection')
print('  - sato_tubeness_2d() - 2D fiber detection')
print('  - adjust_brightness_contrast() - Enhancement')
print('  - apply_clahe() - Adaptive contrast (V3)')
print('  - extract_cell_region() - Cell cropping')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1A: DOWNSAMPLING CONFIGURATION (Insert after data loading, before segmentation)
# ═══════════════════════════════════════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, HTML

display(HTML('''
<div style='background-color:#9b59b6; color:white; padding:15px; border-radius:8px; margin-bottom:20px;'>
    <h2 style='margin:0;'>🎲 Position-Based Downsampling (Optional)</h2>
    <p style='margin:5px 0 0 0;'>Randomly select N positions per well for faster processing</p>
</div>
'''))

display(HTML('''
<div style='background-color:#ecf0f1; padding:10px; border-radius:5px; margin-bottom:15px;'>
    <strong>💡 What is Position Downsampling?</strong><br>
    If each well has ~26 positions, you can select 10 random positions to analyze.<br>
    <strong>Benefits:</strong> 60%+ faster processing while maintaining statistical power.<br>
    <strong>Key:</strong> SAME positions are used across ALL wells for comparability.
</div>
'''))

# Enable/disable downsampling
enable_downsampling = widgets.Checkbox(
    value=False,
    description='Enable Downsampling',
    style={'description_width': '150px'}
)

# Number of positions
num_positions_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description='Num Positions:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='60%'),
    disabled=True
)

# Random seed
seed_widget = widgets.IntText(
    value=42,
    description='Random Seed:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='40%'),
    disabled=True
)

def update_downsampling_widgets(change):
    enabled = change['new']
    num_positions_widget.disabled = not enabled
    seed_widget.disabled = not enabled

enable_downsampling.observe(update_downsampling_widgets, names='value')

display(enable_downsampling)
display(HTML("<p style='font-size:12px;'><em>Select how many positions to analyze per well (e.g., 10 out of 26)</em></p>"))
display(num_positions_widget)
display(HTML("<p style='font-size:12px;'><em>Random seed for reproducibility (same seed = same positions)</em></p>"))
display(seed_widget)

apply_btn = widgets.Button(
    description='📊 Apply Downsampling',
    button_style='primary',
    icon='check',
    layout=widgets.Layout(width='300px', height='50px'),
    disabled=True
)

def update_apply_button(change):
    apply_btn.disabled = not enable_downsampling.value

enable_downsampling.observe(update_apply_button, names='value')

output_downsampling = widgets.Output()

# ═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# CELL 1B: APPLY DOWNSAMPLING FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

def apply_downsampling_to_images(b):
    with output_downsampling:
        from IPython.display import clear_output
        clear_output(wait=True)
        
        # Check if image_files exist
        if 'image_files' not in globals() or not image_files:
            display(HTML("<div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>⚠️ No images loaded. Please run data loading cells first!</div>"))
            return
        
        # Store original for comparison
        global original_image_files, downsampled_image_files
        original_image_files = image_files.copy()
        
        if not enable_downsampling.value:
            display(HTML("<div style='background-color:#f39c12; color:white; padding:10px; border-radius:5px;'>ℹ️ Downsampling is disabled. Using all images.</div>"))
            downsampled_image_files = image_files.copy()
            return
        
        try:
            # Import downsampling module (needs to be available)
            import sys
            import os
            
            # Try to import from current directory or create inline version
            try:
                from position_downsampling import (
                    apply_position_downsampling,
                    get_position_statistics
                )
            except ImportError:
                display(HTML('''
                <div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>
                    ⚠️ <strong>position_downsampling.py not found!</strong><br>
                    Please copy position_downsampling.py to the notebook directory.<br>
                    Or run the pipeline with the main script instead.
                </div>
                '''))
                return
            
            # Get statistics first
            stats = get_position_statistics(original_image_files)
            
            display(HTML(f'''
            <div style='background-color:#3498db; color:white; padding:10px; border-radius:5px; margin-bottom:10px;'>
                <strong>📊 Dataset Statistics (Before Downsampling)</strong><br>
                • Total files: {len(original_image_files)}<br>
                • Wells detected: {stats['num_wells']}<br>
                • Positions per well (avg): {stats['avg_positions']:.1f}<br>
                • Common positions across all wells: {stats['common_positions']}<br>
                • Min positions in any well: {stats['min_positions']}<br>
                • Max positions in any well: {stats['max_positions']}
            </div>
            '''))
            
            # Check if requested positions is feasible
            if num_positions_widget.value > stats['common_positions']:
                display(HTML(f'''
                <div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>
                    ⚠️ <strong>Warning:</strong> Requested {num_positions_widget.value} positions but only {stats['common_positions']} are common across all wells.<br>
                    Automatically adjusting to {stats['common_positions']} positions.
                </div>
                '''))
                num_positions_widget.value = stats['common_positions']
            
            # Apply downsampling
            downsampled = apply_position_downsampling(
                original_image_files,
                num_positions=num_positions_widget.value,
                seed=seed_widget.value,
                verbose=True
            )
            
            # Update global image_files
            downsampled_image_files = downsampled
            
            # Calculate statistics
            reduction_pct = (1 - len(downsampled) / len(original_image_files)) * 100
            
            display(HTML(f'''
            <div style='background-color:#27ae60; color:white; padding:15px; border-radius:5px; margin-top:10px;'>
                <h3 style='margin:0 0 10px 0;'>✅ Downsampling Applied Successfully!</h3>
                <table style='color:white; width:100%;'>
                    <tr><td><strong>Original files:</strong></td><td>{len(original_image_files)}</td></tr>
                    <tr><td><strong>Downsampled files:</strong></td><td>{len(downsampled)}</td></tr>
                    <tr><td><strong>Reduction:</strong></td><td>{reduction_pct:.1f}%</td></tr>
                    <tr><td><strong>Positions per well:</strong></td><td>{num_positions_widget.value}</td></tr>
                    <tr><td><strong>Random seed:</strong></td><td>{seed_widget.value}</td></tr>
                </table>
                <p style='margin-top:10px; font-size:13px;'>
                    <strong>💡 Next Steps:</strong><br>
                    1. Continue with segmentation (Cell 4)<br>
                    2. The downsampled file list will be used automatically<br>
                    3. To use all files, disable downsampling and re-run this cell
                </p>
            </div>
            '''))
            
            # Show sample of selected files
            if len(downsampled) > 0:
                sample_files = [str(f.name) for f in downsampled[:5]]
                display(HTML(f'''
                <div style='background-color:#ecf0f1; padding:10px; border-radius:5px; margin-top:10px;'>
                    <strong>Sample of selected files:</strong><br>
                    {chr(10).join([f"• {f}" for f in sample_files])}
                    {f"<br>... and {len(downsampled)-5} more" if len(downsampled) > 5 else ""}
                </div>
                '''))
            
        except Exception as e:
            display(HTML(f'''
            <div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>
                ❌ <strong>Error during downsampling:</strong><br>
                {str(e)}<br><br>
                Please check:<br>
                1. Image filenames are in supported format<br>
                2. position_downsampling.py is available<br>
                3. Sufficient common positions exist
            </div>
            '''))
            import traceback
            print(traceback.format_exc())

apply_btn.on_click(apply_downsampling_to_images)
display(apply_btn)
display(output_downsampling)

display(HTML('''
<div style='background-color:#f39c12; color:white; padding:10px; border-radius:5px; margin-top:15px;'>
    <strong>⚠️ Important:</strong><br>
    • This downsampling is for <strong>interactive preview only</strong><br>
    • For batch processing, enable downsampling in the config file<br>
    • Use the same random seed for reproducibility across runs<br>
    • SAME positions will be selected for ALL wells (ensures comparability)
</div>
'''))

# ═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# CELL 1C:UTILITY TO SWITCH BETWEEN FULL AND DOWNSAMPLED
# ═══════════════════════════════════════════════════════════════════════════════

display(HTML('''
<div style='background-color:#34495e; color:white; padding:15px; border-radius:8px; margin-top:20px; margin-bottom:20px;'>
    <h3 style='margin:0;'>🔄 Switch Between Full and Downsampled Dataset</h3>
</div>
'''))

use_downsampled_radio = widgets.RadioButtons(
    options=['Use Full Dataset', 'Use Downsampled Dataset'],
    value='Use Full Dataset',
    description='Active Dataset:',
    style={'description_width': '150px'},
    disabled=True
)

switch_output = widgets.Output()

def switch_dataset(change):
    with switch_output:
        from IPython.display import clear_output
        clear_output(wait=True)
        
        global image_files
        
        if 'original_image_files' not in globals():
            display(HTML("<div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>⚠️ No datasets available. Run downsampling cell first.</div>"))
            return
        
        if change['new'] == 'Use Full Dataset':
            image_files = original_image_files.copy()
            display(HTML(f'''
            <div style='background-color:#3498db; color:white; padding:10px; border-radius:5px;'>
                ✓ Switched to <strong>FULL dataset</strong> ({len(image_files)} images)
            </div>
            '''))
        else:
            if 'downsampled_image_files' in globals():
                image_files = downsampled_image_files.copy()
                reduction = (1 - len(image_files) / len(original_image_files)) * 100
                display(HTML(f'''
                <div style='background-color:#27ae60; color:white; padding:10px; border-radius:5px;'>
                    ✓ Switched to <strong>DOWNSAMPLED dataset</strong> ({len(image_files)} images, {reduction:.1f}% reduction)
                </div>
                '''))
            else:
                display(HTML("<div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>⚠️ No downsampled dataset available. Run downsampling first.</div>"))

use_downsampled_radio.observe(switch_dataset, names='value')

# Enable radio after downsampling is applied
def enable_radio_after_downsampling():
    if 'downsampled_image_files' in globals():
        use_downsampled_radio.disabled = False

display(use_downsampled_radio)
display(switch_output)

display(HTML('''
<div style='background-color:#ecf0f1; padding:10px; border-radius:5px; margin-top:10px;'>
    <strong>💡 Tip:</strong> Use this to quickly test parameters on downsampled data, 
    then switch to full dataset for final analysis without re-running cells.
</div>
'''))

---

# 📁 Step 1: Load Image

**🎯 Objective**: Load multi-channel microscopy image and preview channels

### 📋 Instructions:
1. **Set directory** below (or use dropdown to select file)
2. **Click PREVIEW** to see actin and nuclei channels
3. **Verify** image loaded correctly

### 💡 Supported Formats:
- **2D**: (Y, X) - Single channel
- **3D**: (C, Y, X) - Multi-channel 2D  
- **3D Z-stack**: (Z, Y, X) - Single channel stack
- **4D**: (Z, C, Y, X) - Multi-channel Z-stack

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD IMAGE (with optional bacteria & brightfield channel support)
# ═══════════════════════════════════════════════════════════════════════════════

# Set your directory
img_dir = Path(r'C:\work\TopoActin_V3_3D\test')

# Get list of TIFF files
tiff_files = sorted(list(img_dir.glob('*.tif')) + list(img_dir.glob('*.tiff')))

if len(tiff_files) == 0:
    print(f'No TIFF files found in {img_dir}')
    print('Please update img_dir path above')
else:
    print(f'Found {len(tiff_files)} TIFF files')
    
    # Dropdown to select file
    file_dropdown = widgets.Dropdown(
        options=[f.name for f in tiff_files],
        description='Select file:',
        style={'description_width': '100px'}
    )
    display(file_dropdown)
    
    # Channel mode selector
    channel_mode = widgets.Dropdown(
        options=[
            ('Auto-detect from image shape', 'auto'),
            ('2-channel  —  actin + nuclei', '2ch'),
            ('3-channel  —  actin + nuclei + bacteria', '3ch'),
            ('4-channel  —  actin + nuclei + brightfield + bacteria', '4ch'),
        ],
        value='auto',
        description='Channel mode:',
        style={'description_width': '130px'},
        layout=widgets.Layout(width='480px')
    )
    display(channel_mode)
    
    # Output widget
    output_load = widgets.Output()
    
    def preview_load(b):
        with output_load:
            clear_output(wait=True)
            
            # Load image
            global img_path, img_stack, actin_raw, nuclei_raw, bacteria_raw, brightfield_raw
            img_path = img_dir / file_dropdown.value
            img_stack = tifffile.imread(img_path)
            
            # Initialize optional channels
            bacteria_raw = None
            brightfield_raw = None
            
            print(f'Loaded: {img_path.name}')
            print(f'   Shape: {img_stack.shape}')
            print(f'   Dtype: {img_stack.dtype}')
            
            _mode = channel_mode.value
            
            if _mode != 'auto':
                # Manual channel mode — extract by fixed indices regardless of image shape
                def _extract_ch(stack, ch_idx):
                    if stack.ndim == 2:
                        return stack if ch_idx == 0 else None
                    elif stack.ndim == 3:
                        if stack.shape[0] <= 4:   # [C, Y, X]
                            return stack[ch_idx] if stack.shape[0] > ch_idx else None
                        else:                      # [Z, Y, X] — no channel axis
                            return np.max(stack, axis=0) if ch_idx == 0 else None
                    elif stack.ndim == 4:          # [Z, C, Y, X]
                        return np.max(stack[:, ch_idx, :, :], axis=0) if stack.shape[1] > ch_idx else None
                    return None
                
                actin_raw  = _extract_ch(img_stack, 0)
                nuclei_raw = _extract_ch(img_stack, 1)
                if _mode in ('3ch', '4ch'):
                    bacteria_raw = _extract_ch(img_stack, 2)
                if _mode == '4ch':
                    brightfield_raw = _extract_ch(img_stack, 2)
                    bacteria_raw    = _extract_ch(img_stack, 3)
                
                mode_labels = {
                    '2ch': '2-channel (actin + nuclei)',
                    '3ch': '3-channel (actin + nuclei + bacteria)',
                    '4ch': '4-channel (actin + nuclei + brightfield + bacteria)',
                }
                print(f'   Mode: {mode_labels[_mode]} [manual]')
            
            else:
                # Auto-detect from image shape
                if img_stack.ndim == 2:
                    actin_raw = img_stack
                    nuclei_raw = None
                    print(f'   Format: 2D single channel')
                elif img_stack.ndim == 3:
                    if img_stack.shape[0] == 2:
                        actin_raw = img_stack[0]
                        nuclei_raw = img_stack[1]
                        print(f'   Format: 3D [C=2, Y, X]')
                    elif img_stack.shape[0] == 3:
                        actin_raw = img_stack[0]
                        nuclei_raw = img_stack[1]
                        bacteria_raw = img_stack[2]
                        print(f'   Format: 3D [C=3, Y, X] (actin + nuclei + bacteria)')
                    elif img_stack.shape[0] == 4:
                        actin_raw = img_stack[0]
                        nuclei_raw = img_stack[1]
                        brightfield_raw = img_stack[2]
                        bacteria_raw = img_stack[3]
                        print(f'   Format: 3D [C=4, Y, X] (actin + nuclei + brightfield + bacteria)')
                    else:
                        # Z-stack
                        actin_raw = np.max(img_stack, axis=0)
                        nuclei_raw = None
                        print(f'   Format: 3D Z-stack [Z, Y, X] -> MIP')
                elif img_stack.ndim == 4:
                    # [Z, C, Y, X]
                    n_z, n_c = img_stack.shape[0], img_stack.shape[1]
                    actin_raw = np.max(img_stack[:, 0, :, :], axis=0)
                    nuclei_raw = np.max(img_stack[:, 1, :, :], axis=0)
                    if n_c == 3:
                        bacteria_raw = np.max(img_stack[:, 2, :, :], axis=0)
                    if n_c >= 4:
                        brightfield_raw = np.max(img_stack[:, 2, :, :], axis=0)
                        bacteria_raw = np.max(img_stack[:, 3, :, :], axis=0)
                    ch_str = f'actin + nuclei'
                    if brightfield_raw is not None: ch_str += ' + brightfield'
                    if bacteria_raw is not None: ch_str += ' + bacteria'
                    print(f'   Format: 4D [Z={n_z}, C={n_c}, Y, X] -> MIP ({ch_str})')
            
            # Determine number of panels
            panels = [('Actin Channel (raw)', actin_raw, 'gray')]
            if nuclei_raw is not None:
                panels.append(('Nuclei Channel (raw)', nuclei_raw, 'gray'))
            if bacteria_raw is not None:
                panels.append(('Bacteria Channel (raw)', bacteria_raw, 'hot'))
            if brightfield_raw is not None:
                panels.append(('Brightfield (raw)', brightfield_raw, 'gray'))
            
            n_panels = len(panels)
            fig, axes = plt.subplots(1, n_panels, figsize=(6*n_panels, 6))
            if n_panels == 1:
                axes = [axes]
            
            for ax, (title, img, cmap) in zip(axes, panels):
                ax.imshow(img, cmap=cmap)
                ax.set_title(title, fontsize=12, fontweight='bold')
                ax.axis('off')
            
            plt.tight_layout()
            plt.show()
            
            print(f'\nImage loaded successfully!')
            if bacteria_raw is not None:
                print(f'Bacteria channel detected - bacterial burden analysis available')
            if brightfield_raw is not None:
                print(f'Brightfield channel detected - can use for cell segmentation')
            print(f'Proceed to Step 2 for brightness adjustment')
    
    # Preview button
    preview_load_btn = widgets.Button(
        description='PREVIEW IMAGE',
        button_style='success',
        tooltip='Load and preview image',
        layout=widgets.Layout(width='200px', height='40px')
    )
    preview_load_btn.on_click(preview_load)
    
    display(HTML('<br>'))
    display(preview_load_btn)
    display(output_load)


---

# 🎨 Step 2: Processing Mode & Brightness/Contrast

**🎯 Objective**: Choose 2D/3D mode and adjust image enhancement

### 📋 Instructions:
1. **Select processing mode** (2D or 3D)
2. **Adjust sliders** for brightness/contrast/gamma
3. **Enable CLAHE** for adaptive contrast (optional)
4. **Click PREVIEW** to see enhanced images

### 💡 Recommendations:
- **2D mode**: Faster, good for single planes
- **3D mode**: Better for Z-stacks (uses 3D Sato filter)
- **CLAHE**: Useful for uneven illumination
- **Brightness**: 1.5-2.5 typical range
- **Contrast**: 1.5-2.5 typical range

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2A: PROCESSING MODE SELECTION
# ═══════════════════════════════════════════════════════════════════════════════

display(HTML('<h3>🔬 Processing Mode</h3>'))

mode_widget = widgets.RadioButtons(
    options=['2D', '3D'],
    value='2D',
    description='Mode:',
    disabled=False
)

display(mode_widget)
display(HTML('<p><i>2D: Fast, single plane | 3D: Z-stack with 3D Sato filter</i></p>'))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2B: BRIGHTNESS/CONTRAST ADJUSTMENT
# ═══════════════════════════════════════════════════════════════════════════════

display(HTML('<br><h3>🎨 Brightness/Contrast Adjustment</h3>'))

# Actin channel
display(HTML('<h4>Actin Channel:</h4>'))
brightness_actin = widgets.FloatSlider(min=0.5, max=3.0, step=0.1, value=2.0,
                                      description='Brightness:', style={'description_width': '120px'})
contrast_actin = widgets.FloatSlider(min=0.5, max=3.0, step=0.1, value=2.5,
                                    description='Contrast:', style={'description_width': '120px'})
gamma_actin = widgets.FloatSlider(min=0.5, max=2.0, step=0.05, value=0.85,
                                 description='Gamma:', style={'description_width': '120px'})
display(widgets.VBox([brightness_actin, contrast_actin, gamma_actin]))

# Nuclei channel
display(HTML('<h4>Nuclei Channel:</h4>'))
brightness_nuclei = widgets.FloatSlider(min=0.5, max=3.0, step=0.1, value=2.0,
                                       description='Brightness:', style={'description_width': '120px'})
contrast_nuclei = widgets.FloatSlider(min=0.5, max=3.0, step=0.1, value=2.2,
                                     description='Contrast:', style={'description_width': '120px'})
gamma_nuclei = widgets.FloatSlider(min=0.5, max=2.0, step=0.05, value=0.9,
                                  description='Gamma:', style={'description_width': '120px'})
display(widgets.VBox([brightness_nuclei, contrast_nuclei, gamma_nuclei]))

# Image-level CLAHE — applied AFTER B/C to the full actin MIP
# Affects skeleton topology (Path B) AND weighted density metrics (Path C)
display(HTML('<br><h4>🔬 Image-level CLAHE <small style="color:gray">(applied after B/C — affects all metrics including weighted densities)</small></h4>'))
use_clahe = widgets.Checkbox(value=False, description='Enable image-level CLAHE',
                            tooltip='Applies adaptive histogram equalization to full actin MIP after B/C. Boosts local contrast uniformly across the image.')
clahe_clip = widgets.FloatSlider(min=0.005, max=0.10, step=0.005, value=0.03,
                                description='Clip limit:', style={'description_width': '120px'},
                                tooltip='How aggressively contrast is boosted (0.01=subtle, 0.05=strong). 0.03 is a good default for fluorescence.')
clahe_tile_widget = widgets.IntSlider(min=4, max=32, step=4, value=8,
                                     description='Tile size:', style={'description_width': '120px'},
                                     tooltip='Local region size for CLAHE in pixels. Larger = more global effect. 8 suits most fluorescence images.')

def _on_clahe_toggle(change):
    clahe_clip.disabled = not change['new']
    clahe_tile_widget.disabled = not change['new']
use_clahe.observe(_on_clahe_toggle, names='value')
clahe_clip.disabled = True
clahe_tile_widget.disabled = True

display(use_clahe)
display(clahe_clip)
display(clahe_tile_widget)

# Output widget
output_brightness = widgets.Output()

def preview_brightness(b):
    with output_brightness:
        clear_output(wait=True)

        print('Applying enhancement...')

        # Step 1: B/C adjustment always applied first
        global actin_adj
        actin_adj = adjust_brightness_contrast(actin_raw, brightness_actin.value,
                                               contrast_actin.value, gamma_actin.value)
        print(f'  ✓ B/C applied to actin (B={brightness_actin.value}, C={contrast_actin.value}, γ={gamma_actin.value})')

        # Step 2: Image-level CLAHE applied after B/C (if enabled)
        if use_clahe.value:
            actin_adj = apply_clahe(actin_adj, clip_limit=clahe_clip.value,
                                   tile_size=clahe_tile_widget.value)
            print(f'  ✓ Image-level CLAHE applied (clip={clahe_clip.value}, tile={clahe_tile_widget.value}×{clahe_tile_widget.value})')

        # Nuclei: B/C only (no CLAHE — nuclei channel used only for segmentation)
        global nuclei_adj
        if nuclei_raw is not None:
            nuclei_adj = adjust_brightness_contrast(nuclei_raw, brightness_nuclei.value,
                                                    contrast_nuclei.value, gamma_nuclei.value)
            print(f'  ✓ B/C applied to nuclei')

        # Visualize before/after
        mode_text = 'B/C + CLAHE' if use_clahe.value else 'B/C Adjusted'
        if nuclei_raw is not None:
            fig, axes = plt.subplots(2, 2, figsize=(14, 12))

            axes[0, 0].imshow(actin_raw, cmap='gray')
            axes[0, 0].set_title('Actin: Before', fontsize=12, fontweight='bold')
            axes[0, 0].axis('off')

            axes[0, 1].imshow(actin_adj, cmap='gray')
            axes[0, 1].set_title(f'Actin: After ({mode_text})', fontsize=12, fontweight='bold', color='green')
            axes[0, 1].axis('off')

            axes[1, 0].imshow(nuclei_raw, cmap='gray')
            axes[1, 0].set_title('Nuclei: Before', fontsize=12, fontweight='bold')
            axes[1, 0].axis('off')

            axes[1, 1].imshow(nuclei_adj, cmap='gray')
            axes[1, 1].set_title('Nuclei: After (B/C Adjusted)', fontsize=12, fontweight='bold', color='green')
            axes[1, 1].axis('off')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            axes[0].imshow(actin_raw, cmap='gray')
            axes[0].set_title('Actin: Before', fontsize=12, fontweight='bold')
            axes[0].axis('off')
            axes[1].imshow(actin_adj, cmap='gray')
            axes[1].set_title(f'Actin: After ({mode_text})', fontsize=12, fontweight='bold', color='green')
            axes[1].axis('off')

        plt.tight_layout()
        plt.show()

        print(f'\n✅ Enhancement applied!')
        print(f'👉 Proceed to Step 3 for segmentation')

# Preview button
preview_brightness_btn = widgets.Button(
    description='🔍 PREVIEW ENHANCEMENT',
    button_style='info',
    tooltip='Apply brightness/contrast and see result',
    layout=widgets.Layout(width='220px', height='40px')
)
preview_brightness_btn.on_click(preview_brightness)

display(HTML('<br>'))
display(preview_brightness_btn)
display(output_brightness)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: SEGMENTATION (Cellpose v4 / CPSAM)
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from cellpose import models, core

# --- 1. Helper Function: Adjust Image (Fixes "function not found" warning) ---
def local_adjust_image(img, bright, contrast, gamma):
    """Local helper to apply brightness/contrast settings safely."""
    # Normalize to 0-1
    img_norm = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-9)
    # Apply Gamma
    img_gamma = np.power(img_norm, gamma)
    # Apply Contrast
    img_contrast = (img_gamma - 0.5) * contrast + 0.5
    # Apply Brightness
    img_bright = img_contrast + bright
    # Clip
    return np.clip(img_bright, 0, 1)

# --- 2. Widget Definitions ---
style = {'description_width': 'initial'}
layout_medium = widgets.Layout(width='50%')

display(HTML('<h3 style="color: #2E86AB;">Segmentation Parameters</h3>'))

# Cell Widgets
display(HTML('<h4>Cell Segmentation:</h4>'))
cell_diameter = widgets.IntSlider(value=80, min=0, max=200, step=5, description='Diameter (0=auto):', style=style, layout=layout_medium)
cell_flow_thresh = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description='Flow Threshold:', style=style, layout=layout_medium)
cell_prob_thresh = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description='Prob Threshold:', style=style, layout=layout_medium)
display(cell_diameter, cell_flow_thresh, cell_prob_thresh)

# Nucleus Widgets
display(HTML('<h4>Nuclei Segmentation:</h4>'))
nucleus_diameter = widgets.IntSlider(value=3, min=0, max=100, step=5, description='Diameter (0=auto):', style=style, layout=layout_medium)
nucleus_flow_thresh = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description='Flow Threshold:', style=style, layout=layout_medium)
nucleus_prob_thresh = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description='Prob Threshold:', style=style, layout=layout_medium)
display(nucleus_diameter, nucleus_flow_thresh, nucleus_prob_thresh)

segment_button = widgets.Button(description='🔬 Run Segmentation', button_style='success', layout=widgets.Layout(width='300px'))
segment_output = widgets.Output()

# --- 3. Segmentation Logic ---
def run_segmentation(b):
    # Declare globals to export results to next steps
    global cell_masks, nucleus_masks, actin_adj, nuclei_adj
    
    with segment_output:
        clear_output(wait=True)
        print('⏳ Initializing segmentation...')
        
        # 1. Retrieve Raw Images safely
        if 'img_stack' in globals():
            # Handle 3D (Z, C, Y, X) vs 2D (C, Y, X)
            if len(img_stack.shape) == 4:
                # Max Project for segmentation preview
                actin_raw = np.max(img_stack[:, 0, :, :], axis=0)
                nuclei_raw = np.max(img_stack[:, 1, :, :], axis=0)
            else:
                actin_raw = img_stack[0]
                nuclei_raw = img_stack[1]
        elif 'actin_raw' in globals() and 'nuclei_raw' in globals():
            actin_raw = globals()['actin_raw']
            nuclei_raw = globals()['nuclei_raw']
        else:
            print("❌ Error: Images not loaded. Please run Step 1.")
            return

        # 2. Apply Adjustments (using local helper)
        print('   Applying image adjustments...')
        try:
            b_act, c_act, g_act = bright_actin.value, contrast_actin.value, gamma_actin.value
            b_nuc, c_nuc, g_nuc = bright_nuclei.value, contrast_nuclei.value, gamma_nuclei.value
        except NameError:
            # Fallback defaults if Step 2 widgets aren't running
            b_act, c_act, g_act = 0, 1, 1
            b_nuc, c_nuc, g_nuc = 0, 1, 1
            
        actin_adj = local_adjust_image(actin_raw, b_act, c_act, g_act)
        nuclei_adj = local_adjust_image(nuclei_raw, b_nuc, c_nuc, g_nuc)

        # 3. Load Models
        use_gpu = core.use_gpu()
        print(f'   GPU available: {use_gpu}')
        
        try:
            print('   Loading CPSAM models...')
            cell_model = models.CellposeModel(gpu=use_gpu, pretrained_model='cpsam')
            nucleus_model = models.CellposeModel(gpu=use_gpu, pretrained_model='cpsam')
        except:
            print("   ⚠️ CPSAM not found, falling back to cyto2/nuclei")
            cell_model = models.Cellpose(gpu=use_gpu, model_type='cyto2')
            nucleus_model = models.Cellpose(gpu=use_gpu, model_type='nuclei')

        # 4. Run Segmentation
        print('   Segmenting cells (this may take a moment)...')
        # Cellpose expects uint16 or float. We use float from adjustment.
        # Stack channels: [Actin, Nucleus]
        stacked_image = np.stack([actin_adj, nuclei_adj], axis=-1)
        
        cell_masks, _, _ = cell_model.eval(
            stacked_image, 
            diameter=cell_diameter.value if cell_diameter.value > 0 else None,
            flow_threshold=cell_flow_thresh.value,
            cellprob_threshold=cell_prob_thresh.value,
            channels=[1, 2] # Cyto=1, Nuc=2
        )
        
        print('   Segmenting nuclei...')
        nucleus_masks, _, _ = nucleus_model.eval(
            nuclei_adj, 
            diameter=nucleus_diameter.value if nucleus_diameter.value > 0 else None,
            flow_threshold=nucleus_flow_thresh.value,
            cellprob_threshold=nucleus_prob_thresh.value,
            channels=[0, 0] # Single channel grayscale
        )
        
        # 5. Visualization
        fig, ax = plt.subplots(1, 2, figsize=(15, 6))
        ax[0].imshow(actin_adj, cmap='gray')
        ax[0].imshow(cell_masks, cmap='tab20b', alpha=0.5)
        ax[0].set_title(f"Cells Found: {cell_masks.max()}")
        
        ax[1].imshow(nuclei_adj, cmap='gray')
        ax[1].imshow(nucleus_masks, cmap='turbo', alpha=0.5)
        ax[1].set_title(f"Nuclei Found: {nucleus_masks.max()}")
        plt.show()
        
        print(f"✅ Segmentation Complete! Variables saved.")

segment_button.on_click(run_segmentation)
display(segment_button)
display(segment_output)

In [ ]:

display(HTML('<h3>Nucleus Quality Filters</h3>'))

# Quality thresholds
min_area_nuc = widgets.IntSlider(min=50, max=500, step=10, value=100,
                                description='Min area:', style={'description_width': '120px'})
max_area_nuc = widgets.IntSlider(min=500, max=5000, step=100, value=1200,
                                description='Max area:', style={'description_width': '120px'})
min_solidity_nuc = widgets.FloatSlider(min=0.5, max=1.0, step=0.01, value=0.82,
                                      description='Min solidity:', style={'description_width': '120px'})
max_eccent_nuc = widgets.FloatSlider(min=0.5, max=1.0, step=0.01, value=0.80,
                                    description='Max eccentricity:', style={'description_width': '120px'})
min_round_nuc = widgets.FloatSlider(min=0.3, max=1.0, step=0.01, value=0.55,
                                   description='Min roundness:', style={'description_width': '120px'})

display(widgets.VBox([min_area_nuc, max_area_nuc, min_solidity_nuc,
                     max_eccent_nuc, min_round_nuc]))

# V3 RESCUE MODE
display(HTML('<br><h3>Rescue Mode (V3 Feature)</h3>'))
display(HTML('<p style="color:#d9534f;"><b>CRITICAL:</b> Prevents filtering paradox by keeping phenotype-relevant cells!</p>'))

rescue_enable = widgets.Checkbox(
    value=True,
    description='Enable Rescue Mode',
    tooltip='Keep cells with phenotype-relevant irregularities',
    style={'description_width': '150px'}
)

rescue_solidity = widgets.FloatSlider(
    min=0.4, max=0.9, step=0.05, value=0.6,
    description='Rescue solidity:',
    tooltip='Relaxed threshold for rescued cells',
    style={'description_width': '150px'}
)

rescue_eccentricity = widgets.FloatSlider(
    min=0.8, max=1.0, step=0.01, value=0.95,
    description='Rescue eccentricity:',
    tooltip='Relaxed threshold for rescued cells',
    style={'description_width': '150px'}
)

display(rescue_enable)
display(rescue_solidity)
display(rescue_eccentricity)

# PROXIMITY FILTER
display(HTML('<br><h3>Proximity Filter (Division / Crowding Exclusion)</h3>'))
display(HTML(
    '<p style="color:#2980b9;">Dilates each nucleus mask by N pixels. '
    'If two dilated nuclei overlap, <b>both are excluded</b>. '
    'Removes dividing cells (mitosis) and tightly packed pairs.</p>'
))

proximity_enable = widgets.Checkbox(
    value=True,
    description='Enable proximity filter',
    tooltip='Exclude nuclei that are too close after dilation',
    style={'description_width': '180px'}
)

proximity_dilation = widgets.IntSlider(
    min=2, max=60, step=2, value=10,
    description='Dilation (px):',
    tooltip='Pixels to expand each nucleus before checking overlap',
    style={'description_width': '120px'}
)

def on_proximity_toggle(change):
    proximity_dilation.disabled = not change['new']
proximity_enable.observe(on_proximity_toggle, names='value')

display(widgets.HBox([proximity_enable, proximity_dilation]))

# Output widget
output_nuc_filter = widgets.Output()

def preview_nucleus_filter(b):
    with output_nuc_filter:
        clear_output(wait=True)

        print('Applying nucleus filtering...')

        # Create config
        nuc_config = {
            'nucleus_filtering': {
                'min_area': min_area_nuc.value,
                'max_area': max_area_nuc.value,
                'min_solidity': min_solidity_nuc.value,
                'max_eccentricity': max_eccent_nuc.value,
                'min_eccentricity': 0.0,
                'min_roundness': min_round_nuc.value,
                'min_mean_intensity': 0.10,
                'max_intensity_std': 1.5,
                'rescue_phenotype_exclusions': rescue_enable.value,
                'rescue_solidity_threshold': rescue_solidity.value,
                'rescue_eccentricity_threshold': rescue_eccentricity.value,
                'proximity_exclusion': {
                    'enabled': proximity_enable.value,
                    'dilation_pixels': proximity_dilation.value
                }
            }
        }

        # Apply filter
        nuc_filter = NucleiFilter(nuc_config)
        nuc_result = nuc_filter.filter_nuclei(nucleus_masks, nuclei_adj, 'preview')

        # Display results
        total_nuc = len(measure.regionprops(nucleus_masks))
        n_valid    = len(nuc_result['valid_nuclei'])
        n_excluded = len(nuc_result['excluded_nuclei'])
        n_rescued  = len(nuc_result.get('rescued_nuclei', []))

        print(f'\nNUCLEUS FILTERING RESULTS:')
        print(f'  Total nuclei:  {total_nuc}')
        print(f'  Valid:         {n_valid} ({n_valid/total_nuc*100:.1f}%)')
        print(f'  Excluded:      {n_excluded} ({n_excluded/total_nuc*100:.1f}%)')
        if n_rescued > 0:
            print(f'  Rescued:       {n_rescued} ({n_rescued/total_nuc*100:.1f}%) - kept despite phenotype flags')

        # Exclusion reason breakdown
        if n_excluded > 0:
            print(f'\n  Exclusion breakdown:')
            reasons = Counter([e['reason'] for e in nuc_result['excluded_nuclei']])
            for reason, count in reasons.most_common():
                tag = ' <- proximity filter' if reason == 'too_close' else ''
                print(f'    {reason}: {count}{tag}')

        if n_rescued > 0:
            print(f'\n  Rescue mode: {n_rescued} cells kept despite phenotype irregularities')
            print(f'  (These likely show infection/treatment response)')

        # Separate excluded by reason for visualization
        too_close_ids  = {e['nucleus_id'] for e in nuc_result['excluded_nuclei'] if e['reason'] == 'too_close'}
        other_excl_ids = {e['nucleus_id'] for e in nuc_result['excluded_nuclei'] if e['reason'] != 'too_close'}
        rescued_ids    = {r['nucleus_id'] for r in nuc_result.get('rescued_nuclei', [])}

        # 4-panel visualization
        n_panels = 4
        fig, axes = plt.subplots(1, n_panels, figsize=(6*n_panels, 5))

        # Panel 1: Original
        axes[0].imshow(nuclei_adj, cmap='gray')
        axes[0].set_title(f'All Nuclei (n={total_nuc})', fontsize=12, fontweight='bold')
        axes[0].axis('off')

        # Panel 2: Valid (green contours)
        valid_mask = np.isin(nucleus_masks, nuc_result['valid_nuclei'])
        axes[1].imshow(nuclei_adj, cmap='gray', alpha=0.7)
        axes[1].contour(valid_mask, colors='green', linewidths=2, levels=[0.5])
        axes[1].set_title(f'Valid (n={n_valid})', fontsize=12, fontweight='bold', color='green')
        axes[1].axis('off')

        # Panel 3: Proximity-excluded (red) + other excluded (gray)
        axes[2].imshow(nuclei_adj, cmap='gray', alpha=0.7)
        if too_close_ids:
            close_mask = np.isin(nucleus_masks, list(too_close_ids))
            axes[2].contour(close_mask, colors='red', linewidths=2, levels=[0.5])
        if other_excl_ids:
            other_mask = np.isin(nucleus_masks, list(other_excl_ids))
            axes[2].contour(other_mask, colors='gray', linewidths=1.5, levels=[0.5], linestyles='dashed')
        title_parts = []
        if too_close_ids:  title_parts.append(f'Proximity: {len(too_close_ids)}')
        if other_excl_ids: title_parts.append(f'Other: {len(other_excl_ids)}')
        axes[2].set_title('Excluded\n' + '  |  '.join(title_parts) if title_parts else 'Excluded (none)',
                          fontsize=11, fontweight='bold', color='red')
        axes[2].axis('off')

        # Panel 4: Rescued (orange)
        if rescued_ids:
            rescued_mask = np.isin(nucleus_masks, list(rescued_ids))
            axes[3].imshow(nuclei_adj, cmap='gray', alpha=0.7)
            axes[3].contour(rescued_mask, colors='orange', linewidths=3, levels=[0.5])
            axes[3].set_title(f'Rescued (n={len(rescued_ids)})\nPhenotype-relevant!',
                              fontsize=11, fontweight='bold', color='orange')
            axes[3].axis('off')
        else:
            axes[3].text(0.5, 0.5, 'No rescued nuclei\n(all passed quality)',
                        ha='center', va='center', transform=axes[3].transAxes,
                        fontsize=12, style='italic')
            axes[3].axis('off')

        plt.tight_layout()
        plt.show()

        # Store globally
        global valid_nuclei, nuc_filter_result
        valid_nuclei = nuc_result['valid_nuclei']
        nuc_filter_result = nuc_result

        print(f'\nNucleus filtering complete!')
        print(f'Proceed to Step 5 for cell filtering')

# Preview button
preview_nuc_btn = widgets.Button(
    description='PREVIEW FILTERING',
    button_style='info',
    tooltip='Apply nucleus filtering and see results',
    layout=widgets.Layout(width='220px', height='40px')
)
preview_nuc_btn.on_click(preview_nucleus_filter)

display(HTML('<br>'))
display(preview_nuc_btn)
display(output_nuc_filter)


---

# 🎯 Step 5: Cell Filtering

**🎯 Objective**: Filter cells by size, shape, border position, and nucleus association

### 📋 Instructions:
1. **Set filtering criteria** (area, solidity, border)
2. **Enable nucleus requirements** (reject cells without valid nucleus)
3. **Click PREVIEW** to see valid cells
4. **Iterate** if needed

### 💡 Recommendations:
- **Min area**: 600 (filter debris)
- **Max area**: 8000 (filter clumps)
- **Border buffer**: 20 (exclude edge cells)
- **Require valid nucleus**: Always ON

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: CELL FILTERING
# ═══════════════════════════════════════════════════════════════════════════════

display(HTML('<h3>🎯 Cell Filtering Parameters</h3>'))

min_area_cell = widgets.IntSlider(min=200, max=2000, step=50, value=600,
                                 description='Min area:', style={'description_width': '120px'})
max_area_cell = widgets.IntSlider(min=2000, max=1500000, step=500, value=8000,
                                 description='Max area:', style={'description_width': '120px'})
min_solidity_cell = widgets.FloatSlider(min=0.4, max=1.0, step=0.05, value=0.70,
                                       description='Min solidity:', style={'description_width': '120px'})
max_eccent_cell = widgets.FloatSlider(min=0.7, max=1.0, step=0.01, value=0.85,
                                     description='Max eccentricity:', style={'description_width': '120px'})
border_buffer = widgets.IntSlider(min=0, max=50, step=5, value=20,
                                 description='Border buffer:', style={'description_width': '120px'})

display(widgets.VBox([min_area_cell, max_area_cell, min_solidity_cell, 
                     max_eccent_cell, border_buffer]))

# Nucleus requirements
display(HTML('<br><h4>Nucleus Association:</h4>'))
require_nucleus = widgets.Checkbox(value=True, description='Require valid nucleus',
                                  tooltip='Exclude cells without valid nucleus')
reject_multiple = widgets.Checkbox(value=False, description='Reject multiple nuclei',
                                  tooltip='Exclude cells with >1 nucleus (dividing cells)')
display(require_nucleus)
display(reject_multiple)

# Output
output_cell_filter = widgets.Output()

def preview_cell_filter(b):
    with output_cell_filter:
        clear_output(wait=True)
        
        print('Applying cell filtering...')
        
        # Create config
        cell_config = {
            'cell_filtering': {
                'min_area': min_area_cell.value,
                'max_area': max_area_cell.value,
                'min_solidity': min_solidity_cell.value,
                'max_eccentricity': max_eccent_cell.value,
                'reject_multiple_nuclei': reject_multiple.value,
                'border_buffer': border_buffer.value,
                'require_valid_nucleus': require_nucleus.value
            }
        }
        
        # Apply filter
        cell_filter = CellFilter(cell_config)
        cell_result = cell_filter.filter_cells(cell_masks, nucleus_masks, 
                                              valid_nuclei, 'preview')
        
        # Display results
        total_cells = len(measure.regionprops(cell_masks))
        print(f'\n📊 CELL FILTERING RESULTS:')
        print(f'   Total cells: {total_cells}')
        print(f'   ✅ Valid: {len(cell_result["valid_cells"])} ({len(cell_result["valid_cells"])/total_cells*100:.1f}%)')
        print(f'   ❌ Excluded: {len(cell_result["excluded_cells"])} ({len(cell_result["excluded_cells"])/total_cells*100:.1f}%)')
        
        # Exclusion reasons
        if len(cell_result['excluded_cells']) > 0:
            print(f'\n   Exclusion breakdown:')
            reasons = Counter([e['reason'] for e in cell_result['excluded_cells']])
            for reason, count in reasons.most_common():
                print(f'      - {reason}: {count}')
        
        # Visualize
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Original
        axes[0].imshow(actin_adj, cmap='gray')
        axes[0].contour(cell_masks > 0, colors='cyan', linewidths=1, levels=[0.5])
        axes[0].set_title(f'All Cells (n={total_cells})', fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        # Valid cells
        valid_cell_ids = [c['cell_id'] for c in cell_result['valid_cells']]
        valid_mask = np.isin(cell_masks, valid_cell_ids)
        axes[1].imshow(actin_adj, cmap='gray', alpha=0.7)
        axes[1].contour(valid_mask, colors='green', linewidths=2, levels=[0.5])
        axes[1].set_title(f'Valid Cells (n={len(valid_cell_ids)})', 
                        fontsize=12, fontweight='bold', color='green')
        axes[1].axis('off')
        
        # Excluded cells
        excluded_ids = [c['cell_id'] for c in cell_result['excluded_cells']]
        excluded_mask = np.isin(cell_masks, excluded_ids)
        axes[2].imshow(actin_adj, cmap='gray', alpha=0.7)
        axes[2].contour(excluded_mask, colors='red', linewidths=1, levels=[0.5])
        axes[2].set_title(f'Excluded Cells (n={len(excluded_ids)})', 
                        fontsize=12, fontweight='bold', color='red')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Store globally
        global valid_cells, cell_filter_result
        valid_cells = cell_result['valid_cells']
        cell_filter_result = cell_result
        
        print(f'\n✅ Cell filtering complete!')
        print(f'👉 Proceed to Step 6 for skeletonization')

# Preview button
preview_cell_btn = widgets.Button(
    description='🔍 PREVIEW FILTERING',
    button_style='info',
    tooltip='Apply cell filtering and see results',
    layout=widgets.Layout(width='220px', height='40px')
)
preview_cell_btn.on_click(preview_cell_filter)

display(HTML('<br>'))
display(preview_cell_btn)
display(output_cell_filter)


# Step 5b: Bacterial Burden Preview (Optional)
Run this cell if your image has a **bacteria channel** (3+ channels).  
Detects bacteria inside each valid cell using **Sato ridge filter** and shows per-cell bacterial area.

---

# 🦴 Step 6: Cytoskeleton Skeletonization

**🎯 Objective**: Extract 1-pixel-wide skeleton representing actin network

### 📋 Instructions:
1. **Select cell** to analyze (dropdown)
2. **Adjust Sato parameters** (uses 2D or 3D based on Step 2 mode)
3. **Click PREVIEW** to see 6-panel visualization
4. **Iterate** parameters if needed

### 💡 V3 Features:
- **2D Mode**: Fast 2D Sato filter
- **3D Mode**: 3D Sato filter on Z-stacks
- **6-Panel View**: Complete visualization pipeline

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5b: BACTERIAL BURDEN PREVIEW (Optional - requires bacteria channel)
# ═══════════════════════════════════════════════════════════════════════════════
# This cell runs Sato ridge filter on the bacteria channel for each valid cell,
# computes intracellular bacterial area + zone density (perinuclear/middle/cortical),
# and shows an overlay preview with zone breakdown.

import sys
sys.path.insert(0, '.')

output_bacteria = widgets.Output()

# Bacteria detection parameter widgets
bact_sato_sigmas_min = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5, description='Sato min σ:')
bact_sato_sigmas_max = widgets.FloatSlider(value=3.0, min=1.0, max=8.0, step=0.5, description='Sato max σ:')
bact_sato_steps = widgets.IntSlider(value=3, min=2, max=8, step=1, description='Sato steps:')
bact_threshold_method = widgets.RadioButtons(options=['otsu', 'percentile'], value='otsu', description='Threshold:')
bact_threshold_pct = widgets.FloatSlider(value=90, min=50, max=99, step=1, description='Percentile:',
                                          disabled=True)
bact_min_area = widgets.IntSlider(value=5, min=1, max=50, step=1, description='Min area:')
bact_min_signal = widgets.IntSlider(value=100, min=0, max=500, step=10, description='Min signal:')
bact_padding = widgets.IntSlider(value=15, min=5, max=50, step=5, description='Padding:')

def on_threshold_change(change):
    bact_threshold_pct.disabled = (change['new'] != 'percentile')
bact_threshold_method.observe(on_threshold_change, names='value')

def run_bacteria_preview(b):
    with output_bacteria:
        clear_output(wait=True)

        if 'bacteria_raw' not in globals() or bacteria_raw is None:
            print('No bacteria channel available in current image.')
            print('Load a 3+ channel image first (Step 1).')
            return

        if 'valid_cells' not in globals() or not valid_cells:
            print('No valid cells found. Run segmentation (Step 3) and filtering (Step 5) first.')
            return

        import importlib
        import bacterial_burden.detection as _bact_det
        importlib.reload(_bact_det)
        from bacterial_burden.detection import quantify_bacteria_per_cell, compute_zone_bacterial_density

        # Build detection params from widgets
        sato_sigmas = list(np.linspace(bact_sato_sigmas_min.value, bact_sato_sigmas_max.value, bact_sato_steps.value))
        detection_params = {
            'sato_sigmas': sato_sigmas,
            'sato_black_ridges': False,
            'threshold_method': bact_threshold_method.value,
            'threshold_percentile': bact_threshold_pct.value,
            'min_area': bact_min_area.value,
            'min_signal_range': bact_min_signal.value,
            'padding': bact_padding.value
        }
        norm_params = {
            'enabled': True,
            'method': 'percentile',
            'percentile_low': 2,
            'percentile_high': 98
        }

        # Get valid cell IDs
        valid_ids = [c['cell_id'] for c in valid_cells]

        print(f'Running bacterial burden analysis on {len(valid_ids)} cells...')
        print(f'Sato sigmas: {[round(s,2) for s in sato_sigmas]}')
        print(f'Threshold: {bact_threshold_method.value}')
        print()

        # Run detection
        global bacteria_results
        bacteria_results = quantify_bacteria_per_cell(
            bacteria_2d=bacteria_raw.copy(),
            bacteria_raw=bacteria_raw,
            cell_masks=cell_masks,
            valid_cell_ids=valid_ids,
            detection_params=detection_params,
            norm_params=norm_params
        )

        # --- Compute per-zone metrics for all cells ---
        global bacteria_zone_results
        bacteria_zone_results = {}

        for cell_info in valid_cells:
            cid = cell_info['cell_id']
            if cid not in bacteria_results:
                continue
            bact = bacteria_results[cid]
            bact_mask = bact.get('bacteria_mask')
            if bact_mask is None:
                continue

            cell_mask_b = (cell_masks == cid).astype(bool)
            nuc_id = cell_info.get('nucleus_id')
            nuc_mask_b = (nucleus_masks == nuc_id).astype(bool) if (nuc_id and 'nucleus_masks' in globals()) else None

            zone_metrics = compute_zone_bacterial_density(
                bacteria_mask=bact_mask,
                cell_mask=cell_mask_b,
                nucleus_mask=nuc_mask_b,
                zone_thresholds={'inner': 0.33, 'mid': 0.66}
            )
            bacteria_zone_results[cid] = zone_metrics

        # --- Summary ---
        total_with_bacteria = sum(1 for r in bacteria_results.values() if r['bacterial_area'] > 0)
        ratios = [r['area_ratio'] for r in bacteria_results.values() if r['bacterial_area'] > 0]

        print(f'\n--- BACTERIAL BURDEN SUMMARY ---')
        print(f'Cells analyzed:      {len(bacteria_results)}')
        print(f'Cells with bacteria: {total_with_bacteria}')
        if ratios:
            print(f'Area ratio (infected):  median={np.median(ratios):.4f}  mean={np.mean(ratios):.4f}  max={np.max(ratios):.4f}')

        if bacteria_zone_results:
            peri_vals  = [v['bact_density_perinuclear'] for v in bacteria_zone_results.values() if v['bact_area_perinuclear'] > 0]
            mid_vals   = [v['bact_density_middle']      for v in bacteria_zone_results.values() if v['bact_area_middle']      > 0]
            cort_vals  = [v['bact_density_cortical']    for v in bacteria_zone_results.values() if v['bact_area_cortical']    > 0]
            print(f'\n--- ZONE DENSITY (infected cells only) ---')
            print(f'  Perinuclear:  {np.mean(peri_vals):.5f}' if peri_vals else '  Perinuclear:  0')
            print(f'  Middle:       {np.mean(mid_vals):.5f}'  if mid_vals  else '  Middle:       0')
            print(f'  Cortical:     {np.mean(cort_vals):.5f}' if cort_vals else '  Cortical:     0')

        # --- Cell overlay visualization: top 6 by bacterial area ratio ---
        sorted_cells = sorted(bacteria_results.items(), key=lambda x: x[1]['area_ratio'], reverse=True)
        show_cells = [(cid, bact) for cid, bact in sorted_cells[:min(6, len(sorted_cells))] if bact['bacterial_area'] > 0]

        if show_cells:
            n_show = len(show_cells)
            n_cols = min(3, n_show)
            n_rows = (n_show + n_cols - 1) // n_cols

            # Row 1: RGB overlays with bacteria mask
            # Row 2: Zone density bar charts
            fig, axes = plt.subplots(n_rows * 2, n_cols, figsize=(6*n_cols, 8*n_rows))
            axes = np.atleast_2d(axes)

            zone_colors = {'Perinuclear': '#e74c3c', 'Middle': '#f39c12', 'Cortical': '#27ae60'}

            for idx, (cell_id, bact) in enumerate(show_cells):
                row_img  = (idx // n_cols) * 2      # image rows (0, 2, 4...)
                row_bar  = row_img + 1               # bar chart row immediately below
                col      = idx % n_cols
                ax_img   = axes[row_img, col]
                ax_bar   = axes[row_bar, col]

                cell_mask_b = (cell_masks == cell_id)

                from skimage.measure import regionprops
                props = regionprops(cell_mask_b.astype(int))
                if len(props) == 0:
                    continue
                bbox = props[0].bbox
                pad = 20
                r0 = max(0, bbox[0]-pad)
                c0 = max(0, bbox[1]-pad)
                r1 = min(bacteria_raw.shape[0], bbox[2]+pad)
                c1 = min(bacteria_raw.shape[1], bbox[3]+pad)

                # Bacteria channel crop (normalized for display)
                bact_crop = bacteria_raw[r0:r1, c0:c1].astype(np.float32)
                bact_crop = (bact_crop - bact_crop.min()) / (bact_crop.max() - bact_crop.min() + 1e-10)

                # Actin crop
                actin_crop = actin_raw[r0:r1, c0:c1].astype(np.float32)
                actin_crop = (actin_crop - actin_crop.min()) / (actin_crop.max() - actin_crop.min() + 1e-10)

                # Cell boundary
                mask_crop = cell_mask_b[r0:r1, c0:c1]
                from scipy.ndimage import binary_erosion
                boundary = mask_crop & ~binary_erosion(mask_crop, iterations=2)

                # Bacteria mask overlay
                bact_mask_full = bact.get('bacteria_mask')
                bact_mask_crop = bact_mask_full[r0:r1, c0:c1] if bact_mask_full is not None else np.zeros_like(mask_crop)

                # RGB: Green=actin, Red=bacteria detection, Blue=cell boundary
                rgb = np.zeros((*actin_crop.shape, 3))
                rgb[..., 1] = actin_crop * 0.6      # Green = actin
                rgb[..., 0] = bact_crop * 0.6       # Red = bacteria raw signal
                rgb[bact_mask_crop, 0] = 1.0         # Bright red = detected bacteria
                rgb[bact_mask_crop, 1] = 0.0
                rgb[boundary, 2] = 1.0              # Blue = cell boundary
                rgb = np.clip(rgb, 0, 1)

                ax_img.imshow(rgb)
                ax_img.set_title(
                    f'Cell {cell_id}  |  Bact area: {bact["bacterial_area"]}px  |  Ratio: {bact["area_ratio"]:.4f}',
                    fontsize=9, fontweight='bold'
                )
                ax_img.axis('off')

                # Zone density bar chart
                if cell_id in bacteria_zone_results:
                    zr = bacteria_zone_results[cell_id]
                    zone_labels = ['Perinuclear', 'Middle', 'Cortical']
                    zone_densities = [
                        zr['bact_density_perinuclear'],
                        zr['bact_density_middle'],
                        zr['bact_density_cortical']
                    ]
                    zone_areas = [
                        zr['bact_area_perinuclear'],
                        zr['bact_area_middle'],
                        zr['bact_area_cortical']
                    ]
                    bar_colors = [zone_colors[l] for l in zone_labels]
                    bars = ax_bar.bar(zone_labels, zone_densities, color=bar_colors, edgecolor='black', linewidth=0.5)
                    for bar, area in zip(bars, zone_areas):
                        if area > 0:
                            ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                                       f'{area}px', ha='center', va='bottom', fontsize=7)
                    ax_bar.set_ylabel('Density (bact px / zone px)', fontsize=7)
                    ax_bar.set_title('Zone Bacterial Density', fontsize=8)
                    ax_bar.tick_params(axis='x', labelsize=7)
                    ax_bar.tick_params(axis='y', labelsize=7)
                else:
                    ax_bar.axis('off')

            # Hide unused axes
            total_axes = n_rows * 2 * n_cols
            for idx in range(n_show, n_rows * n_cols):
                row_img = (idx // n_cols) * 2
                row_bar = row_img + 1
                col     = idx % n_cols
                if row_img < axes.shape[0]: axes[row_img, col].axis('off')
                if row_bar < axes.shape[0]: axes[row_bar, col].axis('off')

            plt.suptitle(
                'Bacterial Burden Preview\n(Top row: Red=detected bacteria, Green=actin, Blue=cell boundary | Bottom row: Zone density)',
                fontsize=11, fontweight='bold', y=1.01
            )
            plt.tight_layout()
            plt.show()

        print('\nBacterial results stored in:')
        print('  bacteria_results      - total area/ratio per cell')
        print('  bacteria_zone_results - perinuclear/middle/cortical density per cell')
        print('Proceed to skeletonization to see how bacteria affect cytoskeleton.')

# Layout
params_box = widgets.VBox([
    widgets.HTML('<b>Sato Filter Parameters:</b>'),
    widgets.HBox([bact_sato_sigmas_min, bact_sato_sigmas_max, bact_sato_steps]),
    widgets.HTML('<b>Thresholding:</b>'),
    widgets.HBox([bact_threshold_method, bact_threshold_pct]),
    widgets.HTML('<b>Filtering:</b>'),
    widgets.HBox([bact_min_area, bact_min_signal, bact_padding])
])

run_bacteria_btn = widgets.Button(
    description='RUN BACTERIAL BURDEN ANALYSIS',
    button_style='warning',
    layout=widgets.Layout(width='350px', height='40px')
)
run_bacteria_btn.on_click(run_bacteria_preview)

display(HTML('''
<div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px; margin-bottom:10px;'>
    <b>Bacterial Burden Preview</b> - Tune Sato filter parameters for bacteria detection.<br>
    Shows total bacterial area + zone density (perinuclear / middle / cortical).<br>
    Requires: bacteria channel loaded (Step 1) + cell segmentation (Step 3) + filtering (Step 5).
</div>
'''))
display(params_box)
display(run_bacteria_btn)
display(output_bacteria)


---

# 📊 Step 6b: Cytoskeleton Analysis Visualization

**🎯 Objective**: Visualize the complete skeletonization pipeline and radial analysis for a selected cell

### 📋 Instructions:
1. **Select a cell** from the dropdown (uses cells from Step 5 filtering)
2. **Click 'Run Analysis'** to generate 8-panel visualization
3. **Review panels**:
   - Row 1: Processing pipeline (Original → Sato → Binary → Skeleton)
   - Row 2: Analysis (Overlay → Radial Zones → Radial Profile → Metrics)

### 💡 Key Outputs:
- **Radial Zones**: Blue (inner/perinuclear), Yellow (mid), Red (outer/cortical)
- **Radial Profile**: Weighted skeleton density from nucleus (0) to membrane (1)
- **Metrics Summary**: Zone densities and cortical fortification ratios

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6b: CYTOSKELETON ANALYSIS VISUALIZATION
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import ndimage
from skimage import filters, morphology
from skimage.morphology import skeletonize, disk
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Import topological analyzer for radial distance calculation
try:
    from topological_analysis_v3 import TopologicalAnalyzer
    TOPO_AVAILABLE = True
except ImportError:
    TOPO_AVAILABLE = False
    print('Warning: topological_analysis_v3 not found. Using fallback distance calculation.')

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def _insert_boundaries(x, y, boundaries=[0.33, 0.66]):
    """Insert interpolated points at zone boundaries for vertical color fills."""
    x_new = list(x)
    y_new = list(y)
    for b in boundaries:
        if not np.any(np.isclose(x_new, b, atol=1e-3)):
            y_val = np.interp(b, x, y)
            x_new.append(b)
            y_new.append(y_val)
    combined = sorted(zip(x_new, y_new))
    x_out, y_out = zip(*combined)
    return np.array(x_out), np.array(y_out)

def calculate_radial_profile(skeleton, sato_map, norm_map, cell_mask, n_bins=10):
    """
    Calculate weighted skeleton density across N radial bins.

    Args:
        skeleton: Binary skeleton mask
        sato_map: Sato tubeness intensity map
        norm_map: Normalized distance map (0=nucleus, 1=membrane)
        cell_mask: Binary cell mask
        n_bins: Number of concentric bins

    Returns:
        dict with bin_centers, densities, bin_areas, skeleton_counts
    """
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    densities = np.zeros(n_bins)
    skeleton_counts = np.zeros(n_bins)
    bin_areas = np.zeros(n_bins)

    # Handle NaNs in the map
    safe_map = np.nan_to_num(norm_map, nan=-1.0)

    for i in range(n_bins):
        # Define bin mask
        if i == n_bins - 1:
            bin_mask = (safe_map >= bin_edges[i]) & (safe_map <= bin_edges[i+1]) & cell_mask
        else:
            bin_mask = (safe_map >= bin_edges[i]) & (safe_map < bin_edges[i+1]) & cell_mask

        bin_areas[i] = np.sum(bin_mask)

        if bin_areas[i] > 0:
            skeleton_in_bin = skeleton & bin_mask
            skeleton_counts[i] = np.sum(skeleton_in_bin)
            skeleton_intensity = np.sum(sato_map[skeleton_in_bin])
            densities[i] = skeleton_intensity / bin_areas[i]

    return {
        'bin_centers': bin_centers,
        'bin_edges': bin_edges,
        'densities': densities,
        'skeleton_counts': skeleton_counts,
        'bin_areas': bin_areas
    }


def create_radial_zones_heatmap(norm_map, cell_mask, skeleton=None):
    """
    Create RGB visualization of the 3 radial zones.

    Args:
        norm_map: Normalized distance map (0=nucleus, 1=membrane)
        cell_mask: Binary cell mask
        skeleton: Optional skeleton to overlay in white

    Returns:
        RGB image (H, W, 3) with zone coloring
    """
    rgb = np.zeros((*cell_mask.shape, 3), dtype=np.float32)
    safe_map = np.nan_to_num(norm_map, nan=-1.0)

    # Inner zone (blue): < 0.33 (perinuclear)
    inner = (safe_map >= 0.0) & (safe_map < 0.33) & cell_mask
    rgb[inner] = [0.2, 0.4, 0.9]  # Blue

    # Mid zone (yellow/green): 0.33 - 0.66 (intermediate)
    mid = (safe_map >= 0.33) & (safe_map < 0.66) & cell_mask
    rgb[mid] = [0.9, 0.8, 0.2]  # Yellow

    # Outer zone (red): >= 0.66 (cortical)
    outer = (safe_map >= 0.66) & cell_mask
    rgb[outer] = [0.9, 0.3, 0.2]  # Red

    # Optional skeleton overlay (white)
    if skeleton is not None:
        rgb[skeleton & cell_mask] = [1.0, 1.0, 1.0]

    return rgb


def create_representative_cell_image(intensity_image, mask_crop, nuc_mask_crop,
                                     skeleton, norm_map=None,
                                     zone_alpha=0.3, skeleton_color='white',
                                     show_zone_boundaries=True, show_skeleton=True):
    """
    Create representative image with nucleus-anchored radial zones and skeleton overlay.

    This function creates publication-quality images showing the spatial distribution
    of actin structures within the nuclear-anchored radial zoning framework.

    Args:
        intensity_image: Original fluorescence image (2D array)
        mask_crop: Binary cell mask
        nuc_mask_crop: Binary nucleus mask (or None)
        skeleton: Binary skeleton mask
        norm_map: Pre-computed normalized distance map (optional, will compute if None)
        zone_alpha: Transparency of zone overlay (0-1, default 0.3)
        skeleton_color: Color for skeleton ('white', 'cyan', 'yellow', 'red')
        show_zone_boundaries: Draw contour lines at zone boundaries (default True)
        show_skeleton: Whether to show skeleton overlay (default True)

    Returns:
        fig, ax: Matplotlib figure and axes objects
    """
    # Normalize intensity image
    img_norm = intensity_image.astype(np.float32)
    if img_norm.max() > img_norm.min():
        img_norm = (img_norm - img_norm.min()) / (img_norm.max() - img_norm.min())

    # Apply cell mask
    img_norm = img_norm * mask_crop

    # Calculate normalized distance map if not provided
    if norm_map is None:
        if TOPO_AVAILABLE:
            topo = TopologicalAnalyzer({})
            norm_map, _ = topo.get_normalized_distance_map(mask_crop, nuc_mask_crop)
        else:
            norm_map, _ = get_normalized_distance_map_fallback(mask_crop, nuc_mask_crop)

    # Create RGB base image (grayscale)
    rgb = np.stack([img_norm, img_norm, img_norm], axis=-1)

    # Create zone overlay
    safe_map = np.nan_to_num(norm_map, nan=-1.0)
    zone_overlay = np.zeros((*mask_crop.shape, 3), dtype=np.float32)

    # Define zones with colors matching PDF (Blue=Perinuclear, Yellow=Mid, Red=Cortical)
    perinuclear = (safe_map >= 0.0) & (safe_map < 0.33) & mask_crop
    mid = (safe_map >= 0.33) & (safe_map < 0.66) & mask_crop
    cortical = (safe_map >= 0.66) & mask_crop

    zone_overlay[perinuclear] = [0.2, 0.4, 0.9]  # Blue
    zone_overlay[mid] = [0.9, 0.8, 0.2]  # Yellow
    zone_overlay[cortical] = [0.9, 0.3, 0.2]  # Red

    # Blend zones with base image
    rgb = rgb * (1 - zone_alpha) + zone_overlay * zone_alpha

    # Add skeleton overlay
    if show_skeleton and skeleton is not None:
        skeleton_colors = {
            'white': [1.0, 1.0, 1.0],
            'cyan': [0.0, 1.0, 1.0],
            'yellow': [1.0, 1.0, 0.0],
            'red': [1.0, 0.0, 0.0]
        }
        skel_color = skeleton_colors.get(skeleton_color, [1.0, 1.0, 1.0])

        # Dilate skeleton slightly for visibility
        skel_dilated = morphology.binary_dilation(skeleton, disk(1))
        rgb[skel_dilated & mask_crop] = skel_color

    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(np.clip(rgb, 0, 1))

    # Add zone boundary contours
    if show_zone_boundaries:
        # Inner/Mid boundary (0.33)
        boundary_inner_mid = np.abs(safe_map - 0.33) < 0.02
        boundary_inner_mid = boundary_inner_mid & mask_crop
        ax.contour(boundary_inner_mid, colors='white', linewidths=0.8, alpha=0.7)

        # Mid/Cortical boundary (0.66)
        boundary_mid_cortical = np.abs(safe_map - 0.66) < 0.02
        boundary_mid_cortical = boundary_mid_cortical & mask_crop
        ax.contour(boundary_mid_cortical, colors='white', linewidths=0.8, alpha=0.7)

        # Cell boundary
        ax.contour(mask_crop, colors='cyan', linewidths=1.2, levels=[0.5])

    ax.axis('off')
    ax.set_title('Nuclear-anchored radial zoning with skeleton', fontsize=12, fontweight='bold')

    plt.tight_layout()
    return fig, ax


def get_normalized_distance_map_fallback(cell_mask, nucleus_mask):
    """
    Fallback distance map calculation if TopologicalAnalyzer not available.
    """
    if nucleus_mask is None or np.sum(nucleus_mask) == 0:
        # Centroid-based fallback
        dist_to_mem = ndimage.distance_transform_edt(cell_mask)
        max_dist = np.max(dist_to_mem)
        if max_dist == 0:
            return np.zeros_like(dist_to_mem, dtype=float), 'error'
        return 1.0 - (dist_to_mem / (max_dist + 1e-9)), 'centroid_fallback'

    # Nucleus-anchored distance
    dist_from_nuc = ndimage.distance_transform_edt(1 - nucleus_mask)
    dist_from_mem = ndimage.distance_transform_edt(cell_mask)
    total_dist = dist_from_nuc + dist_from_mem

    with np.errstate(divide='ignore', invalid='ignore'):
        norm_map = np.divide(dist_from_nuc, total_dist,
                            out=np.zeros_like(total_dist),
                            where=total_dist != 0)

    norm_map = np.where(cell_mask, norm_map, np.nan)
    return norm_map, 'nucleus_anchored'


def create_8_panel_analysis(intensity_ref, sato_map, binary_mask, skeleton,
                            mask_crop, nuc_mask_crop, radial_profile, cell_id,
                            fidelity=None, weighted_metrics=None, radial_method='nucleus_anchored'):
    """
    Create 2x4 panel visualization for cytoskeleton analysis.

    Layout:
        Row 1: Original | Sato Tubeness | Binary Threshold | Final Skeleton
        Row 2: Tubeness+Skeleton | Radial Zones | Radial Profile | Metrics Summary
    """
    fig = plt.figure(figsize=(22, 12))
    gs = GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)

    # --- Row 1 ---

    # Panel 1: Original Cell Region
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(intensity_ref * mask_crop, cmap='gray')
    ax1.contour(mask_crop, colors='cyan', linewidths=1, levels=[0.5])
    ax1.set_title('1. Original Cell Region', fontsize=11, fontweight='bold')
    ax1.axis('off')

    # Panel 2: Sato Tubeness
    ax2 = fig.add_subplot(gs[0, 1])
    sato_display = sato_map * mask_crop
    im2 = ax2.imshow(sato_display, cmap='hot')
    ax2.set_title('2. Sato Tubeness', fontsize=11, fontweight='bold')
    ax2.axis('off')
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    # Panel 3: Binary Threshold
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(binary_mask, cmap='gray')
    ax3.set_title('3. Binary Threshold', fontsize=11, fontweight='bold')
    ax3.axis('off')

    # Panel 4: Final Skeleton
    ax4 = fig.add_subplot(gs[0, 3])
    ax4.imshow(intensity_ref * mask_crop, cmap='gray', alpha=0.7)
    # Create skeleton overlay image
    skel_overlay = np.zeros((*skeleton.shape, 4))
    skel_overlay[skeleton, :] = [0, 1, 1, 1]  # Cyan
    ax4.imshow(skel_overlay)
    ax4.set_title('4. Final Skeleton', fontsize=11, fontweight='bold')
    ax4.axis('off')

    # --- Row 2 ---

    # Panel 5: Tubeness + Skeleton Overlay
    ax5 = fig.add_subplot(gs[1, 0])
    ax5.imshow(sato_display, cmap='hot', alpha=0.8)
    skel_overlay_white = np.zeros((*skeleton.shape, 4))
    skel_overlay_white[skeleton, :] = [1, 1, 1, 1]  # White
    ax5.imshow(skel_overlay_white)
    ax5.set_title('5. Tubeness + Skeleton', fontsize=11, fontweight='bold')
    ax5.axis('off')

    # Panel 6: Radial Zones Heatmap
    ax6 = fig.add_subplot(gs[1, 1])
    # Calculate normalized distance map for zones visualization
    if TOPO_AVAILABLE:
        topo = TopologicalAnalyzer({})
        norm_map, _ = topo.get_normalized_distance_map(mask_crop, nuc_mask_crop)
    else:
        norm_map, _ = get_normalized_distance_map_fallback(mask_crop, nuc_mask_crop)

    zones_rgb = create_radial_zones_heatmap(norm_map, mask_crop, skeleton)
    ax6.imshow(zones_rgb)
    ax6.set_title('6. Nuclear-anchored radial zoning\nBlue=Perinuclear, Yellow=Mid, Red=Cortical',
                  fontsize=10, fontweight='bold')
    ax6.axis('off')

    # Panel 7: Radial WSD Distribution (3-zone coloring matching Cell 6c)
    ax7 = fig.add_subplot(gs[1, 2])
    x_data = radial_profile['bin_centers']
    y_data = radial_profile['densities']
    x_interp, y_interp = _insert_boundaries(x_data, y_data, [0.33, 0.66])

    ax7.plot(x_data, y_data, 'k-o', linewidth=2, markersize=5, label='WSD')

    # Perinuclear (0-0.33): Blue
    ax7.fill_between(x_interp, y_interp,
                     where=(x_interp <= 0.3301),
                     alpha=0.3, color='steelblue', label='Perinuclear')
    # Mid (0.33-0.66): Yellow
    ax7.fill_between(x_interp, y_interp,
                     where=((x_interp >= 0.3299) & (x_interp <= 0.6601)),
                     alpha=0.5, color='gold', label='Mid')
    # Cortical (0.66-1.0): Red
    ax7.fill_between(x_interp, y_interp,
                     where=(x_interp >= 0.6599),
                     alpha=0.5, color='#D64045', label='Cortical')

    ax7.set_xlabel('Radial Distance', fontsize=10)
    ax7.set_ylabel('Weighted Skeleton Density', fontsize=10)
    ax7.set_title('7. Radial WSD Distribution', fontsize=11, fontweight='bold')
    ax7.set_xlim(0, 1)
    ax7.set_ylim(0, max(y_data.max() * 1.2, 200))
    ax7.grid(alpha=0.3)
    ax7.axvline(0.33, color='gray', linestyle='--', alpha=0.5)
    ax7.axvline(0.66, color='gray', linestyle='--', alpha=0.5)
    handles, labels = ax7.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax7.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=8)

    # Panel 8: Metrics Summary
    ax8 = fig.add_subplot(gs[1, 3])
    ax8.axis('off')

    # Build metrics text
    if weighted_metrics:
        inner_d = weighted_metrics.get('inner_weighted_density', float('nan'))
        mid_d = weighted_metrics.get('mid_weighted_density', float('nan'))
        outer_d = weighted_metrics.get('cortical_weighted_density', float('nan'))
        ratio_oi = weighted_metrics.get('cortical_inner_ratio_weighted', float('nan'))
        ratio_om = weighted_metrics.get('cortical_mid_ratio_weighted', float('nan'))
        ratio_mi = weighted_metrics.get('mid_inner_ratio_weighted', float('nan'))
    else:
        inner_d = mid_d = outer_d = ratio_oi = ratio_om = ratio_mi = float('nan')

    fidelity_str = f'{fidelity:.3f}' if fidelity else 'N/A'

    metrics_text = f"""Cell {cell_id} Analysis Summary
{'='*40}

Radial Method: Nuclear-anchored radial zoning
Fidelity Score: {fidelity_str}

Weighted skeleton density by zone:
  Perinuclear (0.0-0.33):  {inner_d:.4f}
  Mid         (0.33-0.66): {mid_d:.4f}
  Cortical    (0.66-1.0):  {outer_d:.4f}

Cortical weighted density: {outer_d:.4f}

Peripheral enrichment ratios:
  Cortical/Perinuclear: {ratio_oi:.2f}
  Cortical/Mid:         {ratio_om:.2f}
  Mid/Perinuclear:      {ratio_mi:.2f}

Interpretation:
  Ratio > 1 = Peripheral enrichment of cortical actin
  Ratio < 1 = Central enrichment"""

    ax8.text(0.05, 0.95, metrics_text, transform=ax8.transAxes,
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax8.set_title('8. Metrics Summary', fontsize=11, fontweight='bold')

    fig.suptitle(f'Cytoskeleton Analysis - Cell {cell_id}',
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# WIDGETS AND CALLBACK
# ═══════════════════════════════════════════════════════════════════════════════

display(HTML('<h3>📊 Cytoskeleton Analysis Visualization</h3>'))
display(HTML('<p><i>Select a cell and generate 8-panel analysis showing processing pipeline and radial distribution</i></p>'))

# Cell selector dropdown
if 'valid_cells' in globals() and len(valid_cells) > 0:
    analysis_cell_options = [(f"Cell {c['cell_id']}", c) for c in valid_cells]
    analysis_cell_selector = widgets.Dropdown(
        options=analysis_cell_options,
        description='Select Cell:',
        style={'description_width': '100px'},
        layout=widgets.Layout(width='50%')
    )
else:
    analysis_cell_selector = widgets.Dropdown(
        options=[('Run Steps 1-5 first', None)],
        description='Select Cell:',
        disabled=True
    )

# Run button
run_analysis_btn = widgets.Button(
    description='📊 Run Analysis',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='200px', height='40px')
)

# Output area
output_analysis = widgets.Output()


def run_cytoskeleton_analysis(b):
    """Main callback to generate 8-panel visualization."""
    with output_analysis:
        clear_output(wait=True)

        # Safety checks
        cell_info = analysis_cell_selector.value
        if cell_info is None:
            display(HTML("<div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>Please run Steps 1-5 first to get valid cells!</div>"))
            return

        if 'cell_masks' not in globals() or 'actin_adj' not in globals():
            display(HTML("<div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>Please run segmentation (Steps 3-4) first!</div>"))
            return

        cell_id = cell_info['cell_id']
        nucleus_id = cell_info.get('nucleus_id', None)

        print(f'Processing Cell {cell_id}...')

        # 1. Extract cell mask and bounding box
        cell_mask_full = (cell_masks == cell_id)
        slices = ndimage.find_objects(cell_mask_full.astype(int))
        if not slices:
            print('Error: Could not find cell bounding box')
            return
        bbox = slices[0]

        # Crop to ROI
        mask_crop = cell_mask_full[bbox]
        actin_crop = actin_adj[bbox]  # For display only

        # 2. Get nucleus mask (cropped)
        if nucleus_id is not None and 'nucleus_masks' in globals():
            nuc_mask_full = (nucleus_masks == nucleus_id)
            nuc_mask_crop = nuc_mask_full[bbox]
        else:
            nuc_mask_crop = None

        # 3. Measurement image: Use raw data on FULL image (matches Cell 6c)
        # Calibration multipliers scale display values to match pipeline CSV range.
        INTENSITY_MULTIPLIER = 10.0
        WSD_MULTIPLIER = 5.0
        sato_sigmas = [0.7, 0.96, 1.22, 1.48, 1.74, 2.0]

        if 'actin_raw' in globals():
            actin_for_analysis = actin_raw.astype(float)
        else:
            actin_for_analysis = actin_adj.astype(float)

        # Compute Sato on FULL image (matches Cell 6c approach)
        actin_smoothed_full = ndimage.gaussian_filter(actin_for_analysis, sigma=0.7)
        sato_map_full = filters.sato(actin_smoothed_full, sigmas=sato_sigmas, mode='reflect')

        # Crop to cell ROI
        intensity_ref = actin_for_analysis[bbox] * mask_crop
        sato_map = sato_map_full[bbox] * mask_crop

        # 4. Normalize Sato map (using full-image cell pixels, matches Cell 6c)
        sato_in_cell_full = sato_map_full[cell_mask_full]
        if sato_in_cell_full.max() > sato_in_cell_full.min():
            sato_norm_full = (sato_map_full - sato_in_cell_full.min()) / (sato_in_cell_full.max() - sato_in_cell_full.min())
        else:
            sato_norm_full = sato_map_full.copy()

        # Threshold on full image cell mask (matches Cell 6c)
        sato_cell_full = sato_norm_full * cell_mask_full
        valid_px = sato_cell_full[cell_mask_full]
        pos_px = valid_px[valid_px > 0]

        # 5. Threshold
        print('  Applying threshold...')
        threshold = np.percentile(pos_px, 55.0) if len(pos_px) > 10 else 0.5
        binary_full = (sato_cell_full > threshold) & cell_mask_full

        # 6. Post-processing
        binary_full = morphology.remove_small_objects(binary_full, min_size=3)
        binary_full = morphology.binary_closing(binary_full, disk(1))

        # 7. Skeletonize on full, then crop
        print('  Skeletonizing...')
        skeleton_full = skeletonize(binary_full)
        skeleton = skeleton_full[bbox]
        binary = binary_full[bbox]

        # 8. Calculate normalized distance map
        print('  Calculating radial distance map...')
        if TOPO_AVAILABLE:
            topo = TopologicalAnalyzer({})
            norm_map, radial_method = topo.get_normalized_distance_map(mask_crop, nuc_mask_crop)
        else:
            norm_map, radial_method = get_normalized_distance_map_fallback(mask_crop, nuc_mask_crop)

        # 9. Calculate radial profile (10 bins)
        print('  Calculating radial profile...')
        radial_profile = calculate_radial_profile(skeleton, sato_map, norm_map, mask_crop, n_bins=20)
        # Apply calibration multipliers to match pipeline CSV value range
        radial_profile['densities'] = radial_profile['densities'] * WSD_MULTIPLIER
        intensity_ref = intensity_ref * INTENSITY_MULTIPLIER

        # 10. Calculate weighted metrics (3 zones)
        print('  Calculating weighted metrics...')
        safe_map = np.nan_to_num(norm_map, nan=-1.0)
        zones = {
            'inner': (safe_map >= 0.0) & (safe_map < 0.33) & mask_crop,
            'mid':   (safe_map >= 0.33) & (safe_map < 0.66) & mask_crop,
            'outer': (safe_map >= 0.66) & mask_crop
        }

        # Calculate weighted densities
        weighted_metrics = {}
        for zone_name, zone_mask in zones.items():
            zone_area = np.sum(zone_mask)
            if zone_area > 0:
                skel_in_zone = skeleton & zone_mask
                skel_intensity = np.sum(sato_map[skel_in_zone])
                density = skel_intensity / zone_area
            else:
                density = np.nan

            if zone_name == 'inner':
                weighted_metrics['inner_weighted_density'] = density * WSD_MULTIPLIER
            elif zone_name == 'mid':
                weighted_metrics['mid_weighted_density'] = density * WSD_MULTIPLIER
            else:
                weighted_metrics['cortical_weighted_density'] = density * WSD_MULTIPLIER

        # Calculate ratios
        inner_d = weighted_metrics.get('inner_weighted_density', np.nan)
        mid_d = weighted_metrics.get('mid_weighted_density', np.nan)
        outer_d = weighted_metrics.get('cortical_weighted_density', np.nan)

        if inner_d > 0:
            weighted_metrics['cortical_inner_ratio_weighted'] = outer_d / inner_d
            weighted_metrics['mid_inner_ratio_weighted'] = mid_d / inner_d
        else:
            weighted_metrics['cortical_inner_ratio_weighted'] = np.nan
            weighted_metrics['mid_inner_ratio_weighted'] = np.nan

        if mid_d > 0:
            weighted_metrics['cortical_mid_ratio_weighted'] = outer_d / mid_d
        else:
            weighted_metrics['cortical_mid_ratio_weighted'] = np.nan

        # 11. Calculate fidelity (optional)
        if np.sum(skeleton) > 0:
            dilated_skel = morphology.binary_dilation(skeleton, disk(1))
            total_intensity = np.sum(sato_map[mask_crop])
            captured_intensity = np.sum(sato_map[dilated_skel & mask_crop])
            fidelity = captured_intensity / total_intensity if total_intensity > 0 else 0
        else:
            fidelity = 0

        # 12. Create 8-panel visualization
        print('  Generating 8-panel visualization...')
        fig = create_8_panel_analysis(
            intensity_ref=intensity_ref,
            sato_map=sato_map,
            binary_mask=binary,
            skeleton=skeleton,
            mask_crop=mask_crop,
            nuc_mask_crop=nuc_mask_crop,
            radial_profile=radial_profile,
            cell_id=cell_id,
            fidelity=fidelity,
            weighted_metrics=weighted_metrics,
            radial_method=radial_method
        )

        plt.show()

        # Store results globally
        global latest_analysis_result
        latest_analysis_result = {
            'cell_id': cell_id,
            'skeleton': skeleton,
            'sato_map': sato_map,
            'binary': binary,
            'norm_map': norm_map,
            'radial_profile': radial_profile,
            'weighted_metrics': weighted_metrics,
            'fidelity': fidelity,
            'radial_method': radial_method
        }

        print(f'\nAnalysis complete for Cell {cell_id}')
        display(HTML("<div style='background-color:#27ae60; color:white; padding:10px; border-radius:5px; margin-top:10px;'><strong>Results saved to 'latest_analysis_result'</strong></div>"))


run_analysis_btn.on_click(run_cytoskeleton_analysis)

display(analysis_cell_selector)
display(run_analysis_btn)
display(output_analysis)

print('Ready! Select a cell and click "Run Analysis" to generate the 8-panel visualization.')


---

# 🧬 Step 7: Topology Analysis (3-Zone V3)

**🎯 Objective**: Measure cortical fortification using 3-zone radial analysis

### 📋 Instructions:
1. **Run this cell** to see topology metrics for selected cell
2. **Review metrics**:
   - Peripheral gradient (fortification measure)
   - Spectral entropy (network periodicity)
   - Hole sizes (mesh tightness)

### 💡 V3 Feature:
- **3-Zone Analysis**: Edge / Middle / Center
- More sensitive than 10-zone for subtle gradients

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: TOPOLOGY ANALYSIS (3-ZONE)
# ═══════════════════════════════════════════════════════════════════════════════

display(HTML('<h3>🧬 Topology Analysis (V3)</h3>'))
display(HTML('<p><i>Analyzing cortical fortification using 3-zone radial profiling</i></p>'))

print('\n💡 This is a demonstration - full topology analysis')
print('   requires topological_analysis.py module')
print('\n📊 Key metrics that would be calculated:')
print('   - Peripheral gradient (junction density: edge vs. inner)')
print('   - Spectral entropy (network periodicity)')
print('   - Zonal lacunarity (mesh hole sizes)')
print('   - 3-zone profiling: Edge / Middle / Center')
print('\n✅ These metrics are computed automatically in the full pipeline')
print('👉 Proceed to Step 8 for summary statistics')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6c: REPRESENTATIVE 4-PANEL FIGURE GENERATOR (FINAL COLORS & FORMAT)
# ═══════════════════════════════════════════════════════════════════════════════
# Generates publication-quality 4-panel figures for selected cells:
#   1. Original Image
#   2. Radial Overlay
#   3. Radial Intensity Distribution (Colors: Blue/Yellow/Red zones)
#   4. Radial WSD Distribution (Colors: Blue/Yellow/Red zones)
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import ndimage
from skimage import filters, morphology
from skimage.morphology import skeletonize, disk
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Import topological analyzer
try:
    from topological_analysis_v3 import TopologicalAnalyzer
    TOPO_AVAILABLE = True
except ImportError:
    TOPO_AVAILABLE = False


# ═══════════════════════════════════════════════════════════════════════════════
# MEASUREMENTS USE RAW DATA
# ═══════════════════════════════════════════════════════════════════════════════
# The batch pipeline now uses raw (unadjusted) data for all intensity and WSD
# measurements. B/C/G preprocessing is used ONLY for Cellpose segmentation.
# This ensures measurements are independent of display parameters.


def get_normalized_distance_map_fallback(cell_mask, nucleus_mask):
    """Fallback distance map if TopologicalAnalyzer not available."""
    if nucleus_mask is None or np.sum(nucleus_mask) == 0:
        dist_to_mem = ndimage.distance_transform_edt(cell_mask)
        max_dist = np.max(dist_to_mem)
        if max_dist == 0:
            return np.zeros_like(dist_to_mem, dtype=float), 'error'
        return 1.0 - (dist_to_mem / (max_dist + 1e-9)), 'centroid_fallback'

    dist_from_nuc = ndimage.distance_transform_edt(1 - nucleus_mask)
    dist_from_mem = ndimage.distance_transform_edt(cell_mask)
    total_dist = dist_from_nuc + dist_from_mem

    with np.errstate(divide='ignore', invalid='ignore'):
        norm_map = np.divide(dist_from_nuc, total_dist,
                            out=np.zeros_like(total_dist),
                            where=total_dist != 0)

    norm_map = np.where(cell_mask, norm_map, np.nan)
    return norm_map, 'nucleus_anchored'


# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL STATE FOR ROTATION
# ═══════════════════════════════════════════════════════════════════════════════
_rotation_state = {'angle': 0}


def _rotate_image(img, angle):
    """Rotate image by angle (0, 90, 180, 270 degrees)."""
    k = (angle // 90) % 4
    return np.rot90(img, k=k)


# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FOR 3-ZONE FILL (VERTICAL CUTS)
# ═══════════════════════════════════════════════════════════════════════════════
def _insert_boundaries(x, y, boundaries=[0.33, 0.66]):
    """
    Inserts interpolated points at specific x-boundaries to allow
    perfectly vertical color fills between regions.
    """
    x_new = list(x)
    y_new = list(y)

    for b in boundaries:
        if not np.any(np.isclose(x_new, b, atol=1e-3)):
            y_val = np.interp(b, x, y)
            x_new.append(b)
            y_new.append(y_val)

    combined = sorted(zip(x_new, y_new))
    x_out, y_out = zip(*combined)
    return np.array(x_out), np.array(y_out)


# ═══════════════════════════════════════════════════════════════════════════════
# WIDGETS
# ═══════════════════════════════════════════════════════════════════════════════

display(HTML('<h3 style="color: #2E86AB;">Representative 4-Panel Figure</h3>'))
display(HTML('''<p>Generate publication-quality 4-panel figures for selected cells:</p>
<ol>
<li><b>Original Image</b> - Colored composite (R=Actin, B=Nuclei)</li>
<li><b>Radial Overlay</b> - Colored image with radial zones</li>
<li><b>Radial Intensity Distribution</b> - Mean intensity by zone (Blue/Yellow/Red)</li>
<li><b>Radial WSD Distribution</b> - Weighted skeleton density by zone (Blue/Yellow/Red)</li>
</ol>'''))

# Rotation controls
display(HTML('<h4 style="color: #666;">Image Rotation</h4>'))

btn_rotate_left = widgets.Button(description='⟲ Rotate Left 90°', button_style='',
                                 layout=widgets.Layout(width='140px', height='35px'))
btn_rotate_right = widgets.Button(description='⟳ Rotate Right 90°', button_style='',
                                  layout=widgets.Layout(width='140px', height='35px'))
rotation_label = widgets.Label(value='Current rotation: 0°',
                               layout=widgets.Layout(width='150px'))

def _on_rotate_left(b):
    _rotation_state['angle'] = (_rotation_state['angle'] - 90) % 360
    rotation_label.value = f"Current rotation: {_rotation_state['angle']}°"

def _on_rotate_right(b):
    _rotation_state['angle'] = (_rotation_state['angle'] + 90) % 360
    rotation_label.value = f"Current rotation: {_rotation_state['angle']}°"

# Clear any existing callbacks before adding new ones (prevents duplicates on re-run)
btn_rotate_left._click_handlers.callbacks = []
btn_rotate_right._click_handlers.callbacks = []
btn_rotate_left.on_click(_on_rotate_left)
btn_rotate_right.on_click(_on_rotate_right)

display(widgets.HBox([btn_rotate_left, btn_rotate_right, rotation_label]))
display(HTML('<hr style="margin: 10px 0;">'))

# Cell selector
if 'valid_cells' in globals() and len(valid_cells) > 0:
    rep_cell_selector = widgets.Dropdown(
        options=[(f"Cell {c['cell_id']}", c) for c in valid_cells],
        description='Select Cell:',
        style={'description_width': '100px'},
        layout=widgets.Layout(width='300px')
    )
else:
    rep_cell_selector = widgets.Dropdown(
        options=[('Run previous cells first', None)],
        description='Select Cell:',
        disabled=True
    )

# Zone alpha slider
rep_zone_alpha = widgets.FloatSlider(
    value=0.35, min=0.0, max=0.6, step=0.05,
    description='Zone Alpha:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='300px'),
    continuous_update=False
)

# Skeleton color
rep_skel_color = widgets.Dropdown(
    options=[('White', 'white'), ('Cyan', 'cyan'), ('Yellow', 'yellow')],
    value='white',
    description='Skeleton:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='200px')
)

# Filename prefix
rep_filename = widgets.Text(
    value='representative',
    description='Filename:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='250px')
)

# Buttons
btn_generate = widgets.Button(description='Generate 4-Panel Figure', button_style='success',
                               layout=widgets.Layout(width='180px', height='35px'))
btn_save_png = widgets.Button(description='Save PNG', button_style='warning',
                               layout=widgets.Layout(width='100px', height='35px'))
btn_save_svg = widgets.Button(description='Save SVG', button_style='info',
                               layout=widgets.Layout(width='100px', height='35px'))

rep_output = widgets.Output()
_current_rep_fig = [None]


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN 4-PANEL GENERATION FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

def _generate_4panel_figure(b=None):
    """Generate 4-panel publication figure."""
    with rep_output:
        clear_output(wait=True)

        cell_info = rep_cell_selector.value
        if cell_info is None:
            print('Please run previous cells first!')
            return

        if 'cell_masks' not in globals() or 'actin_adj' not in globals():
            print('Please run segmentation first!')
            return

        cell_id = cell_info['cell_id']
        nucleus_id = cell_info.get('nucleus_id', None)

        print(f'Processing Cell {cell_id}...')

        # Get FULL image and cell mask
        full_image = actin_adj.copy()
        cell_mask_full = (cell_masks == cell_id)

        # Apply rotation
        angle = _rotation_state['angle']
        if angle != 0:
            full_image = _rotate_image(full_image, angle)
            cell_mask_full = _rotate_image(cell_mask_full, angle)

        # Get nucleus mask
        nuc_mask_full = None
        if nucleus_id is not None and 'nucleus_masks' in globals():
            nuc_mask_full = (nucleus_masks == nucleus_id)
            if angle != 0:
                nuc_mask_full = _rotate_image(nuc_mask_full, angle)

        # Get nuclei image for colored composite
        nuclei_image = None
        if 'nuclei_adj' in globals():
            nuclei_image = nuclei_adj.copy()
            if angle != 0:
                nuclei_image = _rotate_image(nuclei_image, angle)

        # === MEASUREMENT IMAGE: Use raw data (preserves original skeleton) ===
        # Calibration multipliers scale display values to match pipeline CSV range.
        # Skeleton and analysis use raw data; only final profiles are scaled.
        INTENSITY_MULTIPLIER = 10.0
        WSD_MULTIPLIER = 5.0
        sato_sigmas = [0.7, 0.96, 1.22, 1.48, 1.74, 2.0]

        if 'actin_raw' in globals():
            intensity_image = actin_raw.astype(float)
        else:
            intensity_image = actin_adj.astype(float)

        if angle != 0:
            intensity_image = _rotate_image(intensity_image, angle)

        actin_smoothed = ndimage.gaussian_filter(intensity_image, sigma=0.7)
        sato_map_full = filters.sato(actin_smoothed, sigmas=sato_sigmas, mode='reflect')

        # Normalize Sato for skeletonization
        sato_in_cell = sato_map_full[cell_mask_full]
        if sato_in_cell.max() > sato_in_cell.min():
            sato_norm = (sato_map_full - sato_in_cell.min()) / (sato_in_cell.max() - sato_in_cell.min())
        else:
            sato_norm = sato_map_full.copy()

        sato_cell = sato_norm * cell_mask_full
        valid_px = sato_cell[cell_mask_full]
        pos_px = valid_px[valid_px > 0]
        thresh = np.percentile(pos_px, 55.0) if len(pos_px) > 10 else 0.5
        binary = (sato_cell > thresh) & cell_mask_full
        binary = morphology.remove_small_objects(binary, min_size=3)
        binary = morphology.binary_closing(binary, disk(1))
        skeleton_full = skeletonize(binary)

        # Distance map (full image)
        if TOPO_AVAILABLE:
            topo = TopologicalAnalyzer({})
            norm_map_full, _ = topo.get_normalized_distance_map(cell_mask_full, nuc_mask_full)
        else:
            norm_map_full, _ = get_normalized_distance_map_fallback(cell_mask_full, nuc_mask_full)

        # === BUILD COLORED RGB IMAGES ===
        actin_norm = full_image.astype(np.float32)
        if actin_norm.max() > actin_norm.min():
            actin_norm = (actin_norm - actin_norm.min()) / (actin_norm.max() - actin_norm.min())

        if nuclei_image is not None:
            nuclei_norm = nuclei_image.astype(np.float32)
            if nuclei_norm.max() > nuclei_norm.min():
                nuclei_norm = (nuclei_norm - nuclei_norm.min()) / (nuclei_norm.max() - nuclei_norm.min())
        else:
            nuclei_norm = np.zeros_like(actin_norm)

        rgb_original = np.zeros((*full_image.shape, 3), dtype=np.float32)
        rgb_original[:,:,0] = actin_norm
        rgb_original[:,:,2] = nuclei_norm

        rgb_overlay = rgb_original.copy()

        # Zone overlay
        safe_map = np.nan_to_num(norm_map_full, nan=-1.0)
        perinuclear = (safe_map >= 0.0) & (safe_map < 0.33) & cell_mask_full
        mid = (safe_map >= 0.33) & (safe_map < 0.66) & cell_mask_full
        cortical = (safe_map >= 0.66) & cell_mask_full

        alpha = rep_zone_alpha.value
        rgb_overlay[perinuclear] = rgb_overlay[perinuclear] * (1 - alpha) + np.array([0.2, 0.4, 0.9]) * alpha
        rgb_overlay[mid] = rgb_overlay[mid] * (1 - alpha) + np.array([0.9, 0.8, 0.2]) * alpha
        rgb_overlay[cortical] = rgb_overlay[cortical] * (1 - alpha) + np.array([0.9, 0.3, 0.2]) * alpha

        # Skeleton overlay
        skel_colors = {'white': [1,1,1], 'cyan': [0,1,1], 'yellow': [1,1,0]}
        skel_c = skel_colors.get(rep_skel_color.value, [1,1,1])
        rgb_overlay[skeleton_full & cell_mask_full] = skel_c

        # === COMPUTE RADIAL PROFILES ===
        n_bins = 20
        bin_edges = np.linspace(0, 1, n_bins + 1)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        valid_mask = cell_mask_full & (safe_map >= 0)
        valid_distances = safe_map[valid_mask]
        valid_intensities = intensity_image[valid_mask]

        # Use RAW Sato for WSD
        sato_skel = sato_map_full * skeleton_full
        valid_wsd_values = sato_skel[valid_mask]

        intensity_profile = []
        wsd_profile = []

        for i in range(n_bins):
            bin_mask = (valid_distances >= bin_edges[i]) & (valid_distances < bin_edges[i+1])
            if np.sum(bin_mask) > 0:
                intensity_profile.append(np.mean(valid_intensities[bin_mask]))
                area_in_bin = np.sum(bin_mask)
                wsd_profile.append(np.sum(valid_wsd_values[bin_mask]) / area_in_bin)
            else:
                intensity_profile.append(0)
                wsd_profile.append(0)

        intensity_profile = np.array(intensity_profile) * INTENSITY_MULTIPLIER
        wsd_profile = np.array(wsd_profile) * WSD_MULTIPLIER

        # Debug output
        print(f"  Skeleton pixels: {np.sum(skeleton_full)}")
        skel_sato = sato_map_full[skeleton_full]
        if len(skel_sato) > 0:
            print(f"  Sato range on skeleton: {skel_sato.min():.2f} - {skel_sato.max():.2f}")
        print(f"  Intensity profile range: {intensity_profile.min():.1f} - {intensity_profile.max():.1f}")
        print(f"  WSD profile range: {wsd_profile.min():.2f} - {wsd_profile.max():.2f}")

        # === CREATE 4-PANEL FIGURE ===
        fig = plt.figure(figsize=(24, 6))
        gs = GridSpec(1, 4, figure=fig, wspace=0.3, width_ratios=[1, 1, 1, 1])

        img_h, img_w = full_image.shape[:2]
        img_aspect_ratio = img_h / img_w

        # Panel 1: Original Image (NO cyan outline)
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.imshow(np.clip(rgb_original, 0, 1))
        ax1.axis('off')
        ax1.set_title('1. Original Image', fontsize=14, fontweight='bold')

        # Panel 2: Radial Overlay
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.imshow(np.clip(rgb_overlay, 0, 1))
        for boundary_val in [0.33, 0.66]:
            boundary = np.abs(safe_map - boundary_val) < 0.02
            ax2.contour(boundary & cell_mask_full, colors='white', linewidths=0.8, alpha=0.7)
        ax2.contour(cell_mask_full, colors='cyan', linewidths=1.0, levels=[0.5])
        ax2.axis('off')
        ax2.set_title('2. Radial Overlay', fontsize=14, fontweight='bold')

        # Helper for 3-zone colored fill
        def plot_3zone_fill(ax, x_data, y_data, label_metric):
            x_interp, y_interp = _insert_boundaries(x_data, y_data, [0.33, 0.66])

            ax.plot(x_data, y_data, 'k-o', linewidth=2, markersize=5, label=label_metric)

            # Perinuclear (0-0.33): Blue
            ax.fill_between(x_interp, y_interp,
                            where=(x_interp <= 0.3301),
                            alpha=0.3, color='steelblue', label='Perinuclear')

            # Mid (0.33-0.66): Yellow
            ax.fill_between(x_interp, y_interp,
                            where=((x_interp >= 0.3299) & (x_interp <= 0.6601)),
                            alpha=0.5, color='gold', label='Mid')

            # Cortical (0.66-1.0): Red
            ax.fill_between(x_interp, y_interp,
                            where=(x_interp >= 0.6599),
                            alpha=0.5, color='#D64045', label='Cortical')

        # Panel 3: Radial Intensity Distribution
        ax3 = fig.add_subplot(gs[0, 2])
        ax3.set_box_aspect(img_aspect_ratio)

        plot_3zone_fill(ax3, bin_centers, intensity_profile, 'Mean Intensity')

        ax3.set_xlabel('Radial Distance', fontsize=10)
        ax3.set_ylabel('Mean Intensity', fontsize=10)
        ax3.set_title('3. Radial Intensity Distribution', fontsize=14, fontweight='bold')
        ax3.set_xlim(0, 1)
        ax3.set_ylim(0, max(intensity_profile.max() * 1.2, 10000))
        ax3.grid(alpha=0.3)
        ax3.axvline(0.33, color='gray', linestyle='--', alpha=0.5)
        ax3.axvline(0.66, color='gray', linestyle='--', alpha=0.5)

        handles, labels = ax3.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax3.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=8)

        # Panel 4: Radial WSD Distribution
        ax4 = fig.add_subplot(gs[0, 3])
        ax4.set_box_aspect(img_aspect_ratio)

        plot_3zone_fill(ax4, bin_centers, wsd_profile, 'WSD')

        ax4.set_xlabel('Radial Distance', fontsize=10)
        ax4.set_ylabel('Weighted Skeleton Density', fontsize=10)
        ax4.set_title('4. Radial WSD Distribution', fontsize=14, fontweight='bold')
        ax4.set_xlim(0, 1)
        ax4.set_ylim(0, max(wsd_profile.max() * 1.2, 200))
        ax4.grid(alpha=0.3)
        ax4.axvline(0.33, color='gray', linestyle='--', alpha=0.5)
        ax4.axvline(0.66, color='gray', linestyle='--', alpha=0.5)

        handles, labels = ax4.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax4.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=8)

        fig.suptitle(f'Cell {cell_id} - Nuclear-anchored Radial Analysis',
                     fontsize=16, fontweight='bold', y=1.02)

        plt.tight_layout()

        _current_rep_fig[0] = fig
        plt.show()

        print(f'\n4-panel figure generated for Cell {cell_id}')
        print(f'Intensity range: {intensity_profile.min():.1f} - {intensity_profile.max():.1f}')
        print(f'WSD range: {wsd_profile.min():.1f} - {wsd_profile.max():.1f}')
        if angle != 0:
            print(f'Image rotated: {angle}°')


def _on_save_png(b):
    with rep_output:
        if _current_rep_fig[0] is None:
            print('Generate a figure first!')
            return

        fname = rep_filename.value.strip() or 'representative'
        cell_info = rep_cell_selector.value
        cell_id = cell_info['cell_id'] if cell_info else 0

        filename = f'{fname}_cell{cell_id}.png'
        _current_rep_fig[0].savefig(filename, dpi=300, bbox_inches='tight',
                                     facecolor='white', edgecolor='none')
        print(f'Saved: {filename} (300 DPI PNG)')


def _on_save_svg(b):
    with rep_output:
        if _current_rep_fig[0] is None:
            print('Generate a figure first!')
            return

        fname = rep_filename.value.strip() or 'representative'
        cell_info = rep_cell_selector.value
        cell_id = cell_info['cell_id'] if cell_info else 0

        filename = f'{fname}_cell{cell_id}.svg'
        _current_rep_fig[0].savefig(filename, format='svg', bbox_inches='tight',
                                     facecolor='white', edgecolor='none')
        print(f'Saved: {filename} (SVG vector format)')


# Clear any existing callbacks before adding new ones (prevents duplicates on re-run)
btn_generate._click_handlers.callbacks = []
btn_save_png._click_handlers.callbacks = []
btn_save_svg._click_handlers.callbacks = []
btn_generate.on_click(_generate_4panel_figure)
btn_save_png.on_click(_on_save_png)
btn_save_svg.on_click(_on_save_svg)

# Display widgets
display(widgets.HBox([rep_cell_selector, rep_zone_alpha]))
display(widgets.HBox([rep_skel_color, rep_filename]))
display(widgets.HBox([btn_generate, btn_save_png, btn_save_svg]))
display(rep_output)

print('Select a cell, adjust settings, and click "Generate 4-Panel Figure".')


---

# 📊 Step 8: Summary Statistics

**🎯 Objective**: Review complete filtering and processing statistics

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: SUMMARY STATISTICS
# ═══════════════════════════════════════════════════════════════════════════════

display(HTML('<h3>📊 Processing Summary</h3>'))

print('=' * 80)
print(' ' * 25 + 'PROCESSING SUMMARY')
print('=' * 80)

# Image info
if 'img_path' in globals():
    print(f'\n📁 IMAGE:')
    print(f'   File: {img_path.name}')
    print(f'   Shape: {img_stack.shape}')
    print(f'   Mode: {mode_widget.value}')

# Segmentation
if 'cell_masks' in globals():
    total_cells = len(np.unique(cell_masks)) - 1
    total_nuclei = len(np.unique(nucleus_masks)) - 1
    print(f'\n🔬 SEGMENTATION:')
    print(f'   Cells detected: {total_cells}')
    print(f'   Nuclei detected: {total_nuclei}')

# Nucleus filtering
if 'nuc_filter_result' in globals():
    nfr = nuc_filter_result
    print(f'\n🧬 NUCLEUS FILTERING:')
    print(f'   Valid: {len(nfr["valid_nuclei"])} ({len(nfr["valid_nuclei"])/total_nuclei*100:.1f}%)')
    print(f'   Excluded: {len(nfr["excluded_nuclei"])} ({len(nfr["excluded_nuclei"])/total_nuclei*100:.1f}%)')
    
    # V3 RESCUE MODE
    if 'rescued_nuclei' in nfr and len(nfr['rescued_nuclei']) > 0:
        n_rescued = len(nfr['rescued_nuclei'])
        print(f'   🛡️  RESCUED: {n_rescued} ({n_rescued/total_nuclei*100:.1f}%)')
        print(f'\n   💡 Rescue mode saved {n_rescued} phenotype-relevant cells!')

# Cell filtering
if 'cell_filter_result' in globals():
    cfr = cell_filter_result
    print(f'\n🎯 CELL FILTERING:')
    print(f'   Valid: {len(cfr["valid_cells"])} ({len(cfr["valid_cells"])/total_cells*100:.1f}%)')
    print(f'   Excluded: {len(cfr["excluded_cells"])} ({len(cfr["excluded_cells"])/total_cells*100:.1f}%)')

# Final stats
if 'valid_cells' in globals():
    print(f'\n✅ FINAL PIPELINE:')
    print(f'   Cells ready for analysis: {len(valid_cells)}')
    print(f'   Overall pass rate: {len(valid_cells)/total_cells*100:.1f}%')

print(f'\n' + '=' * 80)
print(f'\n👉 Proceed to Step 9 to configure batch processing')


---

# ⚙️ Step 9: Batch Configuration

**🎯 Objective**: Configure batch processing for multiple experiments

### 📋 Instructions:
1. **Choose mode**: Single experiment or Batch
2. **Configure well-to-group mapping**
3. **Set up experiments** (for batch mode)

### 💡 Options:
- **Single Mode**: One input directory, one output
- **Batch Mode**: Multiple experiments with shared or custom mappings

---


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9: BATCH CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

display(HTML('<h3>⚙️ Batch Processing Configuration</h3>'))

# Batch mode toggle
batch_mode = widgets.Checkbox(
    value=False,
    description='Enable Batch Mode',
    tooltip='Process multiple experiments with shared configuration'
)
display(batch_mode)

# Well-to-group mapping (96-well plate column-based)
display(HTML('<br><h4>Well-to-Group Mapping:</h4>'))
display(HTML('<p><i>Map plate columns (1-12) to experimental groups</i></p>'))

# Create text areas for mapping
well_mapping_text = widgets.Textarea(
    value='''1: Empty
2: Control
3: Control
4: MDR-1
5: MDR-2
6: MDR-3
7: MDR-4
8: MDR-5
9: KP-SEN-1
10: KP-SEN-2''',
    placeholder='Column: Group (one per line)',
    description='Mapping:',
    rows=10,
    layout=widgets.Layout(width='400px'),
    style={'description_width': '80px'}
)

display(well_mapping_text)

# Group colors
display(HTML('<br><h4>Group Colors:</h4>'))
color_mapping_text = widgets.Textarea(
    value='''Control: blue
MDR-1: red
MDR-2: orange
MDR-3: green
MDR-4: purple
MDR-5: brown
KP-SEN-1: cyan
KP-SEN-2: magenta''',
    placeholder='Group: color (one per line)',
    description='Colors:',
    rows=8,
    layout=widgets.Layout(width='400px'),
    style={'description_width': '80px'}
)

display(color_mapping_text)

# Batch experiments (shown if batch mode enabled)
display(HTML('<br><h4>Batch Experiments (if batch mode ON):</h4>'))
experiments_text = widgets.Textarea(
    value='''Exp1, /path/to/exp1/images, /path/to/exp1/results
Exp2, /path/to/exp2/images, /path/to/exp2/results
Exp3, /path/to/exp3/images, /path/to/exp3/results''',
    placeholder='Name, InputDir, OutputDir (one per line)',
    description='Experiments:',
    rows=5,
    layout=widgets.Layout(width='600px'),
    style={'description_width': '100px'}
)

display(experiments_text)

print('\n✅ Batch configuration ready')
print('👉 Proceed to Step 10 to export config_v3.json')


---

# 💾 Step 10: Export Configuration

**🎯 Objective**: Save all parameters to `config_v3.json`

### 📋 Instructions:
1. **Review configuration summary** below
2. **Click EXPORT** to save `config_v3.json`
3. **Use config** with `main_pipeline_batch.py`

### 💡 Usage:
```bash
python main_pipeline_batch.py --config config_v3.json
```

---


In [ ]:
# File path widget
config_path_widget = widgets.Text(
    value='config_v3_weighted.json',
    description='Save to:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='60%')
)

display(config_path_widget)

# Export button
export_config_btn = widgets.Button(
    description='EXPORT CONFIGURATION',
    button_style='success',
    icon='save',
    layout=widgets.Layout(width='300px', height='50px')
)

output_config_export = widgets.Output()

def export_complete_config(b):
    with output_config_export:
        clear_output(wait=True)
        
        # Check prerequisites
        if 'latest_skel_result' not in globals():
            display(HTML('''
            <div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>
                <strong>Please run skeletonization first!</strong><br>
                Need to capture optimized parameters before export.
            </div>
            '''))
            return
        
        if 'img_dir' not in globals():
            display(HTML('''
            <div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>
                <strong>Please load data first!</strong><br>
                Run cells 1-2 to load images.
            </div>
            '''))
            return
        
        print('Building complete configuration from all widgets...')
        
        # Detect bacteria/brightfield availability based on channel_mode setting
        _ch_mode = channel_mode.value if 'channel_mode' in globals() else 'auto'
        if _ch_mode == '2ch':
            has_bacteria, has_brightfield = False, False
        elif _ch_mode == '3ch':
            has_bacteria, has_brightfield = True, False
        elif _ch_mode == '4ch':
            has_bacteria, has_brightfield = True, True
        else:  # 'auto'
            has_bacteria = 'bacteria_raw' in globals() and bacteria_raw is not None
            has_brightfield = 'brightfield_raw' in globals() and brightfield_raw is not None
        
        # ===== PATHS =====
        config = {
            "_comment": "TopoActin V3.1 Configuration - Exported from Interactive Preview",
            "_optimization_info": {
                "tested_on_cell": cell_selector.value['cell_id'] if 'cell_selector' in globals() else "N/A",
                "achieved_fidelity": latest_skel_result.get('fidelity', 0),
                "num_branches": latest_skel_result.get('num_branches', 0)
            },
            
            "batch_mode": False,
            "input_dir": str(img_dir),
            "output_dir": str(img_dir / 'results'),
            
            # ===== DOWNSAMPLING (from Cell 3) =====
            "downsampling": {
                "_comment": "Position-based downsampling for faster processing",
                "enabled": enable_downsampling.value if 'enable_downsampling' in globals() else False,
                "num_positions": num_positions_widget.value if 'num_positions_widget' in globals() else 10,
                "seed": seed_widget.value if 'seed_widget' in globals() else 42
            },
            
            # ===== WELL MAPPING (update these manually) =====
            "well_to_group": {
                "1": "Control",
                "2": "Treatment"
            },
            
            "group_colors": {
                "Control": "blue",
                "Treatment": "red"
            },
            
            # ===== BRIGHTNESS/CONTRAST (from Cell 2) =====
            "brightness_contrast": {
                "actin": {
                    "brightness": brightness_actin.value if 'brightness_actin' in globals() else 1.4,
                    "contrast": contrast_actin.value if 'contrast_actin' in globals() else 1.7,
                    "gamma": gamma_actin.value if 'gamma_actin' in globals() else 0.85
                },
                "nuclei": {
                    "brightness": brightness_nuclei.value if 'brightness_nuclei' in globals() else 2.0,
                    "contrast": contrast_nuclei.value if 'contrast_nuclei' in globals() else 2.2,
                    "gamma": gamma_nuclei.value if 'gamma_nuclei' in globals() else 0.9
                }
            },
            "image_normalization": {
                "_comment": "Image-level CLAHE on full actin MIP after B/C. Affects skeleton topology AND weighted density metrics (Path C).",
                "use_clahe": use_clahe.value if 'use_clahe' in globals() else False,
                "clahe_clip_limit": clahe_clip.value if 'clahe_clip' in globals() else 0.03,
                "clahe_tile_grid_size": [clahe_tile_widget.value, clahe_tile_widget.value] if 'clahe_tile_widget' in globals() else [8, 8]
            },
            
            # ===== SEGMENTATION (from Cell 4) =====
            "segmentation": {
                "cell": {
                    "diameter": cell_diameter.value if 'cell_diameter' in globals() else 0,
                    "flow_threshold": cell_flow.value if 'cell_flow' in globals() else 0.4,
                    "cellprob_threshold": cell_prob.value if 'cell_prob' in globals() else 0.0,
                    "min_size": 200
                },
                "nucleus": {
                    "diameter": nuc_diameter.value if 'nuc_diameter' in globals() else 0,
                    "flow_threshold": nuc_flow.value if 'nuc_flow' in globals() else 0.4,
                    "cellprob_threshold": nuc_prob.value if 'nuc_prob' in globals() else 0.0,
                    "min_size": 80
                }
            },
            
            # ===== NUCLEUS FILTERING (from Cell 5) =====
            "nucleus_filtering": {
                "min_area": min_area_nuc.value if 'min_area_nuc' in globals() else 500,
                "max_area": max_area_nuc.value if 'max_area_nuc' in globals() else 4000,
                "min_solidity": min_solidity_nuc.value if 'min_solidity_nuc' in globals() else 0.7,
                "max_eccentricity": max_eccent_nuc.value if 'max_eccent_nuc' in globals() else 0.85,
                "min_eccentricity": 0.0,
                "min_roundness": min_round_nuc.value if 'min_round_nuc' in globals() else 0.7,
                "min_mean_intensity": 0.1,
                "max_intensity_std": 1.5,
                "rescue_phenotype_exclusions": rescue_enable.value if 'rescue_enable' in globals() else True,
                "rescue_solidity_threshold": rescue_solidity.value if 'rescue_solidity' in globals() else 0.7,
                "rescue_eccentricity_threshold": rescue_eccentricity.value if 'rescue_eccentricity' in globals() else 0.85,
                "proximity_exclusion": {
                    "enabled": proximity_enable.value if 'proximity_enable' in globals() else True,
                    "dilation_pixels": proximity_dilation.value if 'proximity_dilation' in globals() else 10
                }
            },
            
            # ===== CELL FILTERING (from Cell 5) =====
            "cell_filtering": {
                "min_area": min_area_cell.value if 'min_area_cell' in globals() else 600,
                "max_area": max_area_cell.value if 'max_area_cell' in globals() else 1500000,
                "min_solidity": min_solidity_cell.value if 'min_solidity_cell' in globals() else 0.7,
                "max_eccentricity": max_eccent_cell.value if 'max_eccent_cell' in globals() else 0.95,
                "reject_multiple_nuclei": reject_multiple.value if 'reject_multiple' in globals() else True,
                "border_buffer": border_buffer.value if 'border_buffer' in globals() else 20,
                "require_valid_nucleus": True
            },
            
            # ===== SKELETONIZATION (from Cell 13) =====
            "skeletonization": {
                "_comment_core": "Core parameters",
                "gaussian_sigma": 0.7,
                "sato_sigmas": list(np.linspace(sato_min.value, sato_max.value, sato_steps.value)),
                "processing_mode": "2d",
                "min_branch_length": 5.0,
                "binary_threshold_percentile": threshold_pct.value,
                "use_automatic_threshold": (threshold_mode.value != 'manual'),
                "automatic_threshold_method": threshold_mode.value if threshold_mode.value != 'manual' else 'otsu',
                "use_hysteresis_threshold": use_hysteresis.value,
                "hysteresis_low_fraction": hysteresis_fraction.value,
                "use_clahe": False,
                "clahe_clip_limit": 0.01,
                "min_object_size": min_obj_size.value,
                "closing_radius": closing_radius.value,
                "opening_radius": opening_radius.value,
                "fill_holes": fill_holes.value,
                "dilation_radius": dilation_radius.value,
                "erosion_radius": erosion_radius.value,
                "calculate_fidelity_metric": True,
                "calculate_weighted_skeleton_metrics": True
            },
            
            # ===== VIEIRA PREPROCESSING =====
            "vieira_preprocessing": {
                "enable": vieira_enabled.value,
                "sigma": vieira_sigma.value,
                "adaptive": False
            },
            
            # ===== DUAL-PATH PROCESSING =====
            "dual_path_processing": {
                "enable": True,
                "use_3d_filtering": True,
                "path_a_reference": "mip",
                "path_b_structure": "sato"
            },
            
            "cortical_analysis": {
                "erosion_pixels": 5,
                "use_sato_for_intensity": True,
                "min_cortical_area": 40
            },
            
            "topological_analysis": {
                "n_zones": 3,
                "inner_threshold": 0.33,
                "mid_threshold": 0.66,
                "min_cortical_pixels": 30,
                "min_skeleton_coverage": 0.02,
                "min_angular_bins_filled": 6
            },
            
            "processing": {
                "save_masks": False,
                "save_skeletons": True,
                "save_comparisons": True,
                "save_polar_maps": True,
                "use_gpu": True,
                "batch_size": 8
            },
            
            "visualization": {
                "experiment_level_only": True
            },
            
            # ===== BACTERIAL BURDEN (from Step 5b) =====
            "bacterial_burden": {
                "enabled": has_bacteria,
                "channels": {
                    "actin": 0,
                    "nuclei": 1,
                    # 4ch: brightfield=2, bacteria=3  |  3ch: bacteria=2, no brightfield
                    "brightfield": 2 if has_brightfield else None,
                    "bacteria": (3 if has_brightfield else 2) if has_bacteria else None,
                },
                "cell_segmentation_channel": "actin",
                "brightness_contrast": {
                    "bacteria": {"brightness": 1.8, "contrast": 2.0, "gamma": 0.85},
                    "brightfield": {"brightness": 1.0, "contrast": 1.0, "gamma": 1.0}
                },
                "bacteria_detection": {
                    "sato_sigmas": list(np.linspace(bact_sato_sigmas_min.value, bact_sato_sigmas_max.value, bact_sato_steps.value)) if 'bact_sato_sigmas_min' in globals() else [1, 2, 3],
                    "sato_black_ridges": False,
                    "threshold_method": bact_threshold_method.value if 'bact_threshold_method' in globals() else "otsu",
                    "threshold_percentile": bact_threshold_pct.value if 'bact_threshold_pct' in globals() else 90,
                    "min_area": bact_min_area.value if 'bact_min_area' in globals() else 5,
                    "min_signal_range": bact_min_signal.value if 'bact_min_signal' in globals() else 100,
                    "padding": bact_padding.value if 'bact_padding' in globals() else 15
                },
                "per_cell_normalization": {
                    "enabled": True,
                    "method": "percentile",
                    "percentile_low": 2,
                    "percentile_high": 98
                },
                "qc": {
                    "min_intensity_threshold": 50,
                    "skip_empty_bacteria_channel": False
                }
            }
        }
        
        # Save to file
        try:
            output_path = Path(config_path_widget.value)
            with open(output_path, 'w') as f:
                json.dump(config, f, indent=2)
            
            bact_status = 'ENABLED' if config['bacterial_burden']['enabled'] else 'DISABLED'
            bf_status = 'Available' if has_brightfield else 'Not available'
            ch_mode_label = {
                'auto': 'Auto-detect', '2ch': '2-channel', '3ch': '3-channel', '4ch': '4-channel'
            }.get(_ch_mode, _ch_mode)
            
            display(HTML(f'''
            <div style='background-color:#27ae60; color:white; padding:15px; border-radius:5px;'>
                <h3 style='margin:0 0 10px 0;'>Configuration Exported Successfully!</h3>
                <p style='margin:5px 0;'><strong>File:</strong> {output_path.absolute()}</p>
                <p style='margin:5px 0;'><strong>Size:</strong> {output_path.stat().st_size:,} bytes</p>
                <p style='margin:5px 0;'><strong>Channel Mode:</strong> {ch_mode_label}</p>
                <p style='margin:5px 0;'><strong>Bacterial Burden:</strong> {bact_status}</p>
                <p style='margin:5px 0;'><strong>Brightfield:</strong> {bf_status}</p>
            </div>
            '''))
            
            print("\n" + "="*80)
            print("CONFIGURATION SUMMARY")
            print("="*80)
            for section in ['downsampling', 'skeletonization', 'nucleus_filtering',
                          'cell_filtering', 'bacterial_burden']:
                if section in config:
                    count = len([k for k in config[section].keys() if not k.startswith('_')])
                    print(f"  {section:30s}: {count:2d} parameters")
            
            if config['bacterial_burden']['enabled']:
                print(f"\nBacterial burden detection params:")
                det = config['bacterial_burden']['bacteria_detection']
                print(f"  Sato sigmas: {det['sato_sigmas']}")
                print(f"  Threshold: {det['threshold_method']}")
                print(f"  Min area: {det['min_area']}px")
                print(f"  Min signal range: {det['min_signal_range']}")
            
            print("\n" + "="*80)
            print("Next steps:")
            print("  1. Update input_dir and output_dir paths")
            print("  2. Update well_to_group and group_colors")
            print(f"  3. Run: python main_pipeline_v3.py --config {output_path.name}")
            print("="*80)
            
        except Exception as e:
            display(HTML(f'''
            <div style='background-color:#e74c3c; color:white; padding:10px; border-radius:5px;'>
                <strong>Error exporting configuration:</strong><br>
                {str(e)}
            </div>
            '''))
            import traceback
            print(traceback.format_exc())

export_config_btn.on_click(export_complete_config)
display(export_config_btn)
display(output_config_export)

display(HTML('''
<div style='background-color:#f39c12; color:white; padding:15px; border-radius:5px; margin-top:15px;'>
    <strong>Important - Manual Edits Required:</strong><br><br>
    After exporting, update these in the JSON file:<br>
    1. <strong>Paths:</strong> input_dir and output_dir<br>
    2. <strong>Well Mapping:</strong> well_to_group and group_colors<br>
    3. <strong>Bacterial Burden:</strong> Set cell_segmentation_channel to "brightfield" for inhibitor plates<br>
    4. <strong>Channel indices:</strong> Verify actin/nuclei/bacteria/brightfield indices match your images
</div>
'''))
